# Visualization Workflow v1
## Standalone Dictionary & Model Validation Visualizations

**Purpose**: Generate comprehensive visualizations for dictionary fitness, model performance, and topic coherence analysis.

**Independence**: This notebook can run independently by loading outputs from any completed workflow run.

**Date Created**: 2026-01-04

---

## Overview

This notebook generates visualizations for:
1. **Dictionary Fitness**: Weight validation, expansion quality, term clustering
2. **Model Comparison**: Pretrained vs. Domain-adapted vs. Policy-trained
3. **Topic Coherence**: Cluster quality, separation metrics
4. **Chunk Analysis**: Clustering, shifts, scoring patterns
5. **Score Distribution**: Cosine vs. BERTJE comparison
6. **Thesis-Specific**: Temporal trends, document type analysis

---

## Required Data Files

**From Source Workflow** (configured via `SOURCE_WORKFLOW`):
- `Dictionary/Curated_dictionary.csv` - Dictionary terms with weights, categories
- `Cosine_labeling/scores_all_labeled.csv` - Chunk scores from cosine similarity
- `Other_data/chunked_corpus.csv` - Chunk metadata (doc_type, year, text)
- `Model_finetuning/trained_encoder/` - Policy-trained model (if comparing models)
- `Model_finetuning/training_metrics.json` - Training history (optional)
- `BERTJE_predictions/bertje_*_predictions_*.csv` - BERTJE predictions (optional)

**Generated Outputs**:
- `Visuals/` - All visualization files (HTML, PNG, CSV)

---

## Quick Start

1. **Set source workflow path** in Cell 1 (Configuration)
2. **Configure models to compare** (base_cosine, pretrained_bertje, slavery_trained, policy_trained)
3. **Apply metadata filters** if needed (doc_type, year_range, doc_folder)
4. **Run all cells** to generate visualizations

---

In [40]:
# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. SOURCE WORKFLOW PATH
# ============================================================

# Set to the workflow directory containing Dictionary/, Cosine_labeling/, etc.
# This is where your COSINE SCORES and DICTIONARY are located
# Example: "workflow_data/Policy_Slavdict_FT-slavery_slavery_v1"
# Or set to None to use current directory

SOURCE_WORKFLOW = r"C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1"  # Path to workflow with Dictionary/ and Cosine_labeling/

# ============================================================
# 2. MODEL COMPARISON SETTINGS
# ============================================================

COMPARE_MODELS = {
    'base_cosine': True,          # Dictionary-based cosine scores (always available)
    'pretrained_bertje': True,    # GroNLP/bert-base-dutch-cased (downloads from HuggingFace)
    'slavery_trained': True,     # Domain-adapted model (set True if you have it)
    'policy_trained': False        # V10 finetuned model (set True if you have it)
}

# VALIDATION MODEL FOR SINGLE-MODEL VISUALIZATIONS
# Which model to use for visualizations that show only one model
# Set to specific model name ('policy_trained', 'slavery_trained', 'pretrained_bertje')
# Or None to use automatic priority: policy > slavery > pretrained
VALIDATION_MODEL = None  # Options: None, 'policy_trained', 'slavery_trained', 'pretrained_bertje'



# ============================================================
# 3. EXPLICIT MODEL PATHS
# ============================================================
# IMPORTANT: Specify exact paths to trained models
# These models can be from ANY workflow directory (not just SOURCE_WORKFLOW)

MODEL_PATHS = {
    # Pretrained BERTje - downloads automatically from HuggingFace
    'pretrained_bertje': 'GroNLP/bert-base-dutch-cased',

    # Slavery-trained model - set to exact directory path
    # Example: "workflow_data/slavery_Slavdict_pretraining_slavery_v13/Model_finetuning/slavery_domain_encoder"
    'slavery_trained': r"C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Model_finetuning",  # SET THIS to your slavery-trained model path

    # Policy-trained model - set to exact directory path
    # Example: "workflow_data/Policy_Slavdict_FT-slavery_slavery_v1/Model_finetuning/trained_encoder"
    'policy_trained': None    # SET THIS to your policy-trained model path
}

# Example configuration (commented out):
# MODEL_PATHS = {
#     'pretrained_bertje': 'GroNLP/bert-base-dutch-cased',
#     'slavery_trained': 'workflow_data/slavery_Slavdict_pretraining_slavery_v13/Model_finetuning/slavery_domain_encoder',
#     'policy_trained': 'workflow_data/Policy_Slavdict_FT-slavery_slavery_v1/Model_finetuning/trained_encoder'
# }

# ============================================================
# 4. METADATA FILTERS
# ============================================================

# Filter chunks by metadata (None = no filter, use all chunks)
METADATA_FILTERS = {
    'doc_type': None,      # None = all, or list like ['policy', 'report']
    'year_range': None,    # None = all, or tuple like (2015, 2024)
    'doc_folder': None     # None = all, or specific folder name
}

# ============================================================
# 5. VISUALIZATION SETTINGS
# ============================================================

MIN_SCORE_THRESHOLD = 0.7        # Minimum score for "relevant" classification
TOP_N_SHIFTERS = 100             # How many top-shifting chunks to visualize
SAMPLE_SIZE_3D = 1000            # Max chunks for 3D plots (performance limit)
PCA_RANDOM_STATE = 42            # For reproducibility
FIGURE_DPI = 150                 # Figure resolution

# ============================================================
# 6. OUTPUT SETTINGS
# ============================================================

SAVE_INTERACTIVE = True          # Save HTML plots (interactive)
SAVE_STATIC = False              # Save PNG/PDF for thesis (requires kaleido)
SHOW_IN_NOTEBOOK = True          # Display plots inline

# ============================================================
# 7. GLOBAL STATE
# ============================================================

VIZ_AVAILABLE = True  # Will be set to False if critical errors occur

print("="*70)
print("VISUALIZATION WORKFLOW v1 - CONFIGURATION")
print("="*70)
print(f"\nData source (Dictionary & Cosine scores):")
print(f"  {SOURCE_WORKFLOW if SOURCE_WORKFLOW else 'Current directory'}")
print(f"\nModels to compare:")
for model, enabled in COMPARE_MODELS.items():
    status = "✓" if enabled else "⊘"
    path = MODEL_PATHS.get(model, 'Not configured')
    print(f"  {status} {model:20s}: {path if enabled else 'Disabled'}")
print(f"\nVisualization settings:")
print(f"  - Score threshold: {MIN_SCORE_THRESHOLD}")
print(f"  - 3D sample size: {SAMPLE_SIZE_3D}")
print(f"  - PCA random state: {PCA_RANDOM_STATE}")
print(f"\nOutput settings:")
print(f"  - Save interactive (HTML): {SAVE_INTERACTIVE}")
print(f"  - Save static (PNG): {SAVE_STATIC}")
print(f"  - Show in notebook: {SHOW_IN_NOTEBOOK}")
print("="*70)

# VALIDATION MODEL FOR SINGLE-MODEL VISUALIZATIONS
# Which model to use for visualizations that show only one model
# Set to specific model name ('policy_trained', 'slavery_trained', 'pretrained_bertje')
# Or None to use automatic priority: policy > slavery > pretrained
VALIDATION_MODEL = None  # Options: None, 'policy_trained', 'slavery_trained', 'pretrained_bertje'



# VALIDATION MODEL FOR SINGLE-MODEL VISUALIZATIONS
# Which model to use for visualizations that show only one model
# Set to specific model name ('policy_trained', 'slavery_trained', 'pretrained_bertje')
# Or None to use automatic priority: policy > slavery > pretrained



VISUALIZATION WORKFLOW v1 - CONFIGURATION

Data source (Dictionary & Cosine scores):
  C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1

Models to compare:
  ✓ base_cosine         : Not configured
  ✓ pretrained_bertje   : GroNLP/bert-base-dutch-cased
  ✓ slavery_trained     : C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Model_finetuning
  ⊘ policy_trained      : Disabled

Visualization settings:
  - Score threshold: 0.7
  - 3D sample size: 1000
  - PCA random state: 42

Output settings:
  - Save interactive (HTML): True
  - Save static (PNG): False
  - Show in notebook: True


In [ ]:
# ============================================================
# FILESYSTEM SETUP
# ============================================================

print(f"\n{'='*70}")
print("FILESYSTEM SETUP")
print(f"{'='*70}")

# Determine base directory
if SOURCE_WORKFLOW is None:
    base_dir = Path.cwd()
    print(f"Source: Current directory")
else:
    base_dir = Path(SOURCE_WORKFLOW)
    if not base_dir.is_absolute():
        base_dir = Path.cwd() / base_dir
    print(f"Source: {base_dir}")

print(f"Base directory: {base_dir}")

# Define folder structure
folders = {
    'Dictionary': base_dir / 'Dictionary',
    'Cosine_labeling': base_dir / 'Cosine_labeling',
    'Bertje_labeling': base_dir / 'Bertje_labeling',  # Added for BERTJE predictions
    'Other_data': base_dir / 'Other_data',
    'Model_finetuning': base_dir / 'Model_finetuning',
    'BERTJE_predictions': base_dir / 'BERTJE_predictions',
    'Visuals': base_dir / 'Visuals'
}

# Check required folders
required_folders = ['Dictionary', 'Cosine_labeling', 'Other_data']
missing_required = []

print(f"\nChecking folder structure:")
for name, path in folders.items():
    exists = path.exists()
    is_required = name in required_folders

    if exists:
        print(f"  {name:20s}: {path} (exists)")
    else:
        if is_required:
            print(f"  {name:20s}: {path} (MISSING - REQUIRED)")
            missing_required.append(name)
        else:
            print(f"  {name:20s}: {path} (not found - optional)")

if missing_required:
    print(f"\nERROR: Missing required folders: {', '.join(missing_required)}")
    print(f"VIZ_AVAILABLE will be set to False")
    VIZ_AVAILABLE = False
else:
    print(f"\nAll required folders found")

# Auto-create Visuals directory
visuals_dir = folders['Visuals']
if not visuals_dir.exists():
    visuals_dir.mkdir(parents=True, exist_ok=True)
    print(f"\nCreated output directory: {visuals_dir}")
else:
    print(f"\nOutput directory ready: {visuals_dir}")

print(f"{'='*70}")



FILESYSTEM SETUP
Source: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1
Base directory: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1

Checking folder structure:
  Dictionary          : C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Dictionary (exists)
  Cosine_labeling     : C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Cosine_labeling (exists)
  Bertje_labeling     : C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Bertje_labeling (exists)
  Other_data          : C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Other_data (exists)
  Model_finetuning    : C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Model_finetun

In [12]:
# ============================================================
# CELL 9.2: IMPORT VISUALIZATION LIBRARIES
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("LOADING VISUALIZATION LIBRARIES")
    print(f"{'='*70}")

    try:
        # Core data handling
        import pandas as pd
        import numpy as np
        from pathlib import Path
        import json
        from typing import Dict, List, Tuple, Optional

        print("\n✓ Core libraries loaded")

        # Visualization
        import matplotlib.pyplot as plt
        import seaborn as sns
        import plotly.graph_objects as go
        import plotly.express as px
        from plotly.subplots import make_subplots

        # Configure plotting defaults
        sns.set_style("whitegrid")
        plt.rcParams['figure.figsize'] = (14, 8)
        plt.rcParams['figure.dpi'] = FIGURE_DPI

        print("✓ Visualization libraries loaded (matplotlib, seaborn, plotly)")

        # ML & Metrics
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import StandardScaler
        from sklearn.metrics import (
            silhouette_score,
            calinski_harabasz_score,
            confusion_matrix
        )
        from sklearn.metrics.pairwise import cosine_similarity
        from scipy.stats import pearsonr, spearmanr

        print("✓ ML & metrics libraries loaded (sklearn, scipy)")

        # NLP - only if models will be compared
        if any(COMPARE_MODELS[k] for k in ['pretrained_bertje', 'slavery_trained', 'policy_trained']):
            from transformers import AutoModel, AutoTokenizer
            import torch
            from tqdm.auto import tqdm

            # Set device
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            print(f"✓ NLP libraries loaded (transformers, torch)")
            print(f"  Device: {device}")
        else:
            print("⊘ NLP libraries skipped (no BERTJE models enabled)")
            device = None

        print(f"\n{'='*70}")
        print("All libraries loaded successfully!")
        print(f"{'='*70}")

    except Exception as e:
        print(f"\n❌ ERROR loading libraries: {e}")
        VIZ_AVAILABLE = False
else:
    print("⚠ Skipping library imports - VIZ_AVAILABLE = False")


LOADING VISUALIZATION LIBRARIES

✓ Core libraries loaded
✓ Visualization libraries loaded (matplotlib, seaborn, plotly)
✓ ML & metrics libraries loaded (sklearn, scipy)
✓ NLP libraries loaded (transformers, torch)
  Device: cuda

All libraries loaded successfully!


In [13]:
# ============================================================
# CELL 9.3: LOAD CORE DATA
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("LOADING CORE DATA")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Load Dictionary
    # --------------------------------------------------------

    print("\n1. Loading dictionary...")

    dict_path = folders['Dictionary'] / 'Curated_dictionary.csv'

    if not dict_path.exists():
        print(f"   ERROR: Dictionary not found at {dict_path}")
        VIZ_AVAILABLE = False
    else:
        df_dict = pd.read_csv(dict_path)

        # Validate expected columns
        required_cols = ['topic', 'term', 'weight', 'category']
        missing_cols = [c for c in required_cols if c not in df_dict.columns]

        if missing_cols:
            print(f"   ERROR: Missing columns: {missing_cols}")
            VIZ_AVAILABLE = False
        else:
            # Extract topics
            topics = sorted(df_dict['topic'].unique().tolist())

            # Check for is_seed column
            has_is_seed = 'is_seed' in df_dict.columns

            print(f"   Loaded {len(df_dict)} dictionary terms")
            print(f"   Topics: {len(topics)}")

            if has_is_seed:
                n_seeds = df_dict['is_seed'].sum()
                n_expanded = len(df_dict) - n_seeds
                print(f"     - Seeds: {n_seeds}")
                print(f"     - Expanded: {n_expanded}")

            # Distribution by topic
            print(f"\n   Terms per topic:")
            for topic in topics:
                count = (df_dict['topic'] == topic).sum()
                print(f"     {topic:50s}: {count:4d}")

if VIZ_AVAILABLE:
    # --------------------------------------------------------
    # 2. Load BERTJE Labeled Corpus
    # --------------------------------------------------------

    print("\n2. Loading BERTJE labeled corpus...")

    bertje_path = folders.get('Bertje_labeling')
    if bertje_path is None:
        bertje_path = base_dir / 'Bertje_labeling'

    bertje_file = bertje_path / 'bertje_labeled_corpus.csv'

    if not bertje_file.exists():
        print(f"   WARNING: BERTJE labels not found at {bertje_file}")
        print(f"   Continuing without BERTJE predictions...")
        df_bertje = None
    else:
        df_bertje = pd.read_csv(bertje_file)
        print(f"   Loaded {len(df_bertje)} BERTJE predictions")
        print(f"   Columns: {list(df_bertje.columns)}")

if VIZ_AVAILABLE:
    # --------------------------------------------------------
    # 3. Load Cosine Scores
    # --------------------------------------------------------

    print("\n3. Loading cosine scores...")

    cosine_path = folders['Cosine_labeling'] / 'scores_all_labeled.csv'

    if not cosine_path.exists():
        print(f"   ERROR: Cosine scores not found at {cosine_path}")
        VIZ_AVAILABLE = False
    else:
        df_cosine = pd.read_csv(cosine_path)

        # Identify score columns
        score_cols = [c for c in df_cosine.columns if c.startswith('score_')]

        if len(score_cols) != len(topics):
            print(f"   WARNING: Score columns ({len(score_cols)}) != topics ({len(topics)})")

        print(f"   Loaded {len(df_cosine)} scored chunks")
        print(f"   Columns: {list(df_cosine.columns)}")

if VIZ_AVAILABLE:
    # --------------------------------------------------------
    # 4. Load Chunked Corpus (for metadata)
    # --------------------------------------------------------

    print("\n4. Loading chunked corpus...")

    chunks_path = folders['Other_data'] / 'chunked_corpus.csv'

    if not chunks_path.exists():
        print(f"   ERROR: Chunked corpus not found at {chunks_path}")
        VIZ_AVAILABLE = False
    else:
        df_chunks = pd.read_csv(chunks_path)

        # Check for metadata columns
        metadata_cols = ['doc_type', 'year', 'document_folder', 'filename']
        available_metadata = [c for c in metadata_cols if c in df_chunks.columns]

        print(f"   Loaded {len(df_chunks)} chunks")
        print(f"   Available metadata: {', '.join(available_metadata)}")

        # Check for text columns
        if 'text_for_scoring' in df_chunks.columns:
            text_col = 'text_for_scoring'
        elif 'raw_text' in df_chunks.columns:
            text_col = 'raw_text'
            print(f"   WARNING: Using raw_text (text_for_scoring not found)")
        else:
            print(f"   ERROR: No text column found")
            VIZ_AVAILABLE = False
            text_col = None

if VIZ_AVAILABLE:
    # --------------------------------------------------------
    # 5. Merge All Data (3-way merge)
    # --------------------------------------------------------

    print("\n5. Merging all data sources...")

    # Start with cosine scores as base
    df_merged = df_cosine.copy()
    print(f"   Base: {len(df_merged)} rows from cosine scores")
    print(f"   Base columns: {len(df_merged.columns)}")

    # Merge with chunked corpus (for metadata and text)
    merge_key = None
    if 'chunk_id' in df_merged.columns and 'chunk_uid' in df_chunks.columns:
        merge_key = ('chunk_id', 'chunk_uid')
        print(f"   Merging with chunked_corpus on chunk_id/chunk_uid...")
        df_merged = df_merged.merge(
            df_chunks,
            left_on='chunk_id',
            right_on='chunk_uid',
            how='left',
            suffixes=('', '_chunks')
        )
    elif 'filename' in df_merged.columns and 'filename' in df_chunks.columns:
        merge_key = ('filename', 'filename')
        print(f"   Merging with chunked_corpus on filename...")
        df_merged = df_merged.merge(
            df_chunks,
            on='filename',
            how='left',
            suffixes=('', '_chunks')
        )
    else:
        print(f"   WARNING: Cannot merge with chunked_corpus - no common key")

    print(f"   After chunked_corpus merge: {len(df_merged)} rows, {len(df_merged.columns)} columns")

    # Merge with BERTJE predictions if available
    if df_bertje is not None:
        # df_bertje has chunk_uid, df_merged has chunk_id (from scores_all_labeled.csv)
        if 'chunk_id' in df_merged.columns and 'chunk_uid' in df_bertje.columns:
            print(f"   Merging with BERTJE predictions on chunk_id/chunk_uid...")
            df_merged = df_merged.merge(
                df_bertje,
                left_on='chunk_id',
                right_on='chunk_uid',
                how='left',
                suffixes=('', '_bertje')
            )
        elif 'chunk_uid' in df_merged.columns and 'chunk_uid' in df_bertje.columns:
            print(f"   Merging with BERTJE predictions on chunk_uid...")
            df_merged = df_merged.merge(
                df_bertje,
                on='chunk_uid',
                how='left',
                suffixes=('', '_bertje')
            )
        elif 'filename' in df_merged.columns and 'filename' in df_bertje.columns:
            print(f"   WARNING: Merging with BERTJE on filename (less precise than chunk-level)...")
            df_merged = df_merged.merge(
                df_bertje,
                on='filename',
                how='left',
                suffixes=('', '_bertje')
            )
        else:
            print(f"   WARNING: Cannot merge with BERTJE - no common key")
            print(f"   df_merged keys: {[c for c in df_merged.columns if 'chunk' in c or c == 'filename']}")
            print(f"   df_bertje keys: {[c for c in df_bertje.columns if 'chunk' in c or c == 'filename']}")

        print(f"   After BERTJE merge: {len(df_merged)} rows, {len(df_merged.columns)} columns")

    # Replace df_cosine with fully merged version
    df_cosine = df_merged

    print(f"   Final merged data: {len(df_cosine)} rows, {len(df_cosine.columns)} columns")
    print(f"   All columns preserved from all sources")

if VIZ_AVAILABLE:
    # --------------------------------------------------------
    # 6. Apply Metadata Filters (if configured)
    # --------------------------------------------------------

    print("\n6. Applying metadata filters...")

    original_count = len(df_cosine)

    # Filter by doc_type
    if METADATA_FILTERS['doc_type'] is not None and 'doc_type' in df_cosine.columns:
        df_cosine = df_cosine[df_cosine['doc_type'].isin(METADATA_FILTERS['doc_type'])]
        print(f"   Filtered by doc_type: {original_count} -> {len(df_cosine)}")
        original_count = len(df_cosine)

    # Filter by year range
    if METADATA_FILTERS['year_range'] is not None and 'year' in df_cosine.columns:
        min_year, max_year = METADATA_FILTERS['year_range']
        df_cosine = df_cosine[
            (df_cosine['year'] >= min_year) & (df_cosine['year'] <= max_year)
        ]
        print(f"   Filtered by year: {original_count} -> {len(df_cosine)}")
        original_count = len(df_cosine)

    # Filter by document folder
    if METADATA_FILTERS['doc_folder'] is not None and 'document_folder' in df_cosine.columns:
        df_cosine = df_cosine[df_cosine['document_folder'] == METADATA_FILTERS['doc_folder']]
        print(f"   Filtered by doc_folder: {original_count} -> {len(df_cosine)}")

    if len(df_cosine) == original_count:
        print(f"   No filters applied (using all {len(df_cosine)} chunks)")

    if len(df_cosine) == 0:
        print(f"   ERROR: No chunks left after filtering!")
        VIZ_AVAILABLE = False

if VIZ_AVAILABLE:
    # --------------------------------------------------------
    # 7. Create Visuals Output Directory
    # --------------------------------------------------------

    print("\n7. Creating output directory...")

    visuals_dir = folders.get('Visuals')
    if visuals_dir is None:
        visuals_dir = base_dir / 'Visuals'

    visuals_dir.mkdir(parents=True, exist_ok=True)

    print(f"   Output directory: {visuals_dir}")

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print(f"\n{'='*70}")
    print("DATA LOADING COMPLETE")
    print(f"{'='*70}")
    print(f"Dictionary: {len(df_dict)} terms across {len(topics)} topics")
    print(f"Scored chunks: {len(df_cosine)}")
    print(f"Total columns: {len(df_cosine.columns)}")
    print(f"Topics: {topics}")
    print(f"Output: {visuals_dir}")
    print(f"{'='*70}")
    print(f"\nSection 1 Complete - Ready for model loading!")

else:
    print(f"\nData loading failed - cannot proceed with visualizations")



LOADING CORE DATA

1. Loading dictionary...
   Loaded 782 dictionary terms
   Topics: 3
     - Seeds: 143
     - Expanded: 639

   Terms per topic:
     Contemporary_Manifestations                       :  203
     Historical_Slavery_Colonialism                    :  296
     Structural_Continuity_Neocolonial                 :  283

2. Loading BERTJE labeled corpus...
   Loaded 2840 BERTJE predictions
   Columns: ['file_path', 'chunk_uid', 'raw_text', 'sentence_count', 'token_count', 'doc_type', 'year', 'document_folder', 'filename', 'text_for_scoring', 'bertje_score_Contemporary_Manifestations', 'bertje_score_Historical_Slavery_Colonialism', 'bertje_score_Structural_Continuity_Neocolonial', 'bertje_primary_topic', 'bertje_max_score', 'bertje_score_margin', 'bertje_primary_score', 'bertje_cv', 'bertje_confidence']

3. Loading cosine scores...
   Loaded 2840 scored chunks
   Columns: ['filename', 'chunk_id', 'sentence_count', 'raw_text', 'text_for_scoring', 'score_Contemporary_Manifest

In [14]:
# ============================================================
# CELL 9.4: LOAD BERTJE MODELS
# ============================================================

print(f"\n{'='*70}")
print("LOADING BERTJE MODELS")
print(f"{'='*70}")

models = {}
tokenizers = {}

# ============================================================
# 1. Base Cosine (No model needed)
# ============================================================

if COMPARE_MODELS['base_cosine']:
    print(f"\n1️⃣  Base Cosine")
    print(f"   ✓ Using existing cosine scores (no model loading needed)")
    models['base_cosine'] = None
    tokenizers['base_cosine'] = None

# ============================================================
# 2. Pretrained BERTje
# ============================================================

if COMPARE_MODELS['pretrained_bertje']:
    print(f"\n2️⃣  Pretrained BERTje")
    print(f"   Path: {MODEL_PATHS['pretrained_bertje']}")

    try:
        from transformers import AutoTokenizer, AutoModel
        import torch

        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATHS['pretrained_bertje'])
        model = AutoModel.from_pretrained(MODEL_PATHS['pretrained_bertje'])
        model.eval()

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)

        models['pretrained_bertje'] = model
        tokenizers['pretrained_bertje'] = tokenizer

        print(f"   ✓ Loaded successfully")
        print(f"   ✓ Device: {device}")

    except Exception as e:
        print(f"   ❌ Failed to load: {str(e)}")
        COMPARE_MODELS['pretrained_bertje'] = False

# ============================================================
# 3. Slavery-trained Model
# ============================================================

if COMPARE_MODELS['slavery_trained']:
    print(f"\n3️⃣  Slavery-trained Model")

    if MODEL_PATHS['slavery_trained'] is not None:
        model_finetuning_path = Path(MODEL_PATHS['slavery_trained'])
        if not model_finetuning_path.is_absolute():
            model_finetuning_path = Path.cwd() / model_finetuning_path

        print(f"   Model_finetuning folder: {model_finetuning_path}")

        # Look for model in this priority order:
        # 1. trained_encoder/ (sentence-transformers format for embeddings)
        # 2. SBERTContinuousMultiLabel/ (continuous regression model)
        # 3. full_model/ (classification model)

        model_candidates = [
            model_finetuning_path / 'trained_encoder',
            model_finetuning_path / 'SBERTContinuousMultiLabel',
            model_finetuning_path / 'full_model'
        ]

        slavery_model_path = None
        for candidate in model_candidates:
            if candidate.exists():
                # Check if it has required files
                if (candidate / 'config.json').exists():
                    slavery_model_path = candidate
                    print(f"   ✓ Found model: {candidate.name}/")
                    break

        if slavery_model_path is None:
            print(f"   ❌ No compatible model found in Model_finetuning folder")
            print(f"   Looked for: trained_encoder/, SBERTContinuousMultiLabel/, full_model/")
            COMPARE_MODELS['slavery_trained'] = False
        else:
            try:
                from transformers import AutoTokenizer, AutoModel
                import torch

                tokenizer = AutoTokenizer.from_pretrained(str(slavery_model_path))
                model = AutoModel.from_pretrained(str(slavery_model_path))
                model.eval()

                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                model = model.to(device)

                models['slavery_trained'] = model
                tokenizers['slavery_trained'] = tokenizer

                print(f"   ✓ Loaded successfully from: {slavery_model_path.name}/")
                print(f"   ✓ Device: {device}")

            except Exception as e:
                print(f"   ❌ Failed to load: {str(e)}")
                COMPARE_MODELS['slavery_trained'] = False
    else:
        print(f"   ⚠ slavery_trained enabled but MODEL_PATHS['slavery_trained'] is None")
        print(f"   Set MODEL_PATHS['slavery_trained'] to Model_finetuning folder path")
        COMPARE_MODELS['slavery_trained'] = False

# ============================================================
# 4. Policy-trained Model
# ============================================================

if COMPARE_MODELS['policy_trained']:
    print(f"\n4️⃣  Policy-trained Model")

    if MODEL_PATHS['policy_trained'] is not None:
        model_finetuning_path = Path(MODEL_PATHS['policy_trained'])
        if not model_finetuning_path.is_absolute():
            model_finetuning_path = Path.cwd() / model_finetuning_path

        print(f"   Model_finetuning folder: {model_finetuning_path}")

        # Look for model in this priority order:
        model_candidates = [
            model_finetuning_path / 'trained_encoder',
            model_finetuning_path / 'SBERTContinuousMultiLabel',
            model_finetuning_path / 'full_model'
        ]

        policy_model_path = None
        for candidate in model_candidates:
            if candidate.exists():
                if (candidate / 'config.json').exists():
                    policy_model_path = candidate
                    print(f"   ✓ Found model: {candidate.name}/")
                    break

        if policy_model_path is None:
            print(f"   ❌ No compatible model found in Model_finetuning folder")
            print(f"   Looked for: trained_encoder/, SBERTContinuousMultiLabel/, full_model/")
            COMPARE_MODELS['policy_trained'] = False
        else:
            try:
                from transformers import AutoTokenizer, AutoModel
                import torch

                tokenizer = AutoTokenizer.from_pretrained(str(policy_model_path))
                model = AutoModel.from_pretrained(str(policy_model_path))
                model.eval()

                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                model = model.to(device)

                models['policy_trained'] = model
                tokenizers['policy_trained'] = tokenizer

                print(f"   ✓ Loaded successfully from: {policy_model_path.name}/")
                print(f"   ✓ Device: {device}")

            except Exception as e:
                print(f"   ❌ Failed to load: {str(e)}")
                COMPARE_MODELS['policy_trained'] = False
    else:
        print(f"   ⚠ policy_trained enabled but MODEL_PATHS['policy_trained'] is None")
        print(f"   Set MODEL_PATHS['policy_trained'] to Model_finetuning folder path")
        COMPARE_MODELS['policy_trained'] = False

# ============================================================
# Summary
# ============================================================

print(f"\n{'='*70}")
print("MODEL LOADING SUMMARY")
print(f"{'='*70}")

loaded_models = [name for name, model in models.items() if model is not None or name == 'base_cosine']
failed_models = [name for name in COMPARE_MODELS.keys()
                if COMPARE_MODELS.get(name, False) and name not in loaded_models]

if loaded_models:
    print(f"✓ Successfully loaded: {len(loaded_models)}")
    for model_name in loaded_models:
        print(f"  - {model_name}")

if failed_models:
    print(f"\n❌ Failed to load: {len(failed_models)}")
    for model_name in failed_models:
        print(f"  - {model_name}")

embedding_models = [name for name in loaded_models if name != 'base_cosine']
print(f"\n✓ Embedding models available: {len(embedding_models)}")

print(f"{'='*70}")


LOADING BERTJE MODELS

1️⃣  Base Cosine
   ✓ Using existing cosine scores (no model loading needed)

2️⃣  Pretrained BERTje
   Path: GroNLP/bert-base-dutch-cased


Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   ✓ Loaded successfully
   ✓ Device: cuda

3️⃣  Slavery-trained Model
   Model_finetuning folder: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Model_finetuning
   ✓ Found model: trained_encoder/
   ✓ Loaded successfully from: trained_encoder/
   ✓ Device: cuda

MODEL LOADING SUMMARY
✓ Successfully loaded: 3
  - base_cosine
  - pretrained_bertje
  - slavery_trained

✓ Embedding models available: 2


In [15]:
# ============================================================
# CELL 9.5: GENERATE DICTIONARY TERM EMBEDDINGS
# ============================================================

if VIZ_AVAILABLE and len([m for m in models.keys() if m != 'base_cosine']) > 0:
    print(f"\n{'='*70}")
    print("GENERATING DICTIONARY TERM EMBEDDINGS")
    print(f"{'='*70}")

    # Initialize storage
    dict_embeddings = {}

    # Get ALL dictionary entries (with topic context)
    # Format: "Topic: term" to provide context
    dict_texts = []
    for idx, row in df_dict.iterrows():
        # Create contextual text: "Topic: term"
        # This ensures the same term gets different embeddings in different topics
        contextual_text = f"{row['topic']}: {row['term']}"
        dict_texts.append(contextual_text)

    n_entries = len(dict_texts)

    print(f"\nDictionary entries to embed: {n_entries}")
    print(f"Format: 'Topic: term' (provides context)")
    print(f"Example: '{dict_texts[0]}'")

    # --------------------------------------------------------
    # Helper Function: Generate Embeddings
    # --------------------------------------------------------

    def generate_embeddings(texts, model, tokenizer, batch_size=32, max_length=128):
        """
        Generate embeddings for a list of texts using mean pooling.

        Args:
            texts: List of strings
            model: HuggingFace model
            tokenizer: HuggingFace tokenizer
            batch_size: Batch size for processing
            max_length: Max token length

        Returns:
            embeddings: numpy array [N, hidden_dim]
        """
        embeddings = []

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
                batch = texts[i:i+batch_size]

                # Tokenize
                inputs = tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors='pt'
                )

                # Move to device
                device = next(model.parameters()).device
                inputs = {k: v.to(device) for k, v in inputs.items()}

                # Forward pass
                outputs = model(**inputs)

                # Mean pooling (average over sequence length, ignoring padding)
                attention_mask = inputs['attention_mask']
                hidden = outputs.last_hidden_state  # [batch, seq_len, hidden_dim]

                # Mask out padding tokens
                mask_expanded = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
                sum_hidden = torch.sum(hidden * mask_expanded, dim=1)
                sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
                mean_pooled = sum_hidden / sum_mask  # [batch, hidden_dim]

                embeddings.append(mean_pooled.cpu().numpy())

        return np.vstack(embeddings)

    # --------------------------------------------------------
    # Generate Embeddings for Each Model
    # --------------------------------------------------------

    for model_name in ['pretrained_bertje', 'slavery_trained', 'policy_trained']:
        if COMPARE_MODELS.get(model_name, False) and model_name in models:
            print(f"\n{model_name}:")
            print(f"  Generating embeddings for {n_entries} entries...")

            try:
                embeddings = generate_embeddings(
                    dict_texts,
                    models[model_name],
                    tokenizers[model_name],
                    batch_size=32,
                    max_length=128  # "Topic: term" is short, 128 tokens sufficient
                )

                dict_embeddings[model_name] = embeddings

                print(f"  Generated: {embeddings.shape}")
                print(f"  Shape: [{n_entries} entries x {embeddings.shape[1]} dimensions]")
                print(f"  Each term embedded WITH its topic context")

            except Exception as e:
                print(f"  ERROR: {e}")
                import traceback
                traceback.print_exc()
                # Don't add to dict_embeddings

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print(f"\n{'='*70}")
    print("DICTIONARY EMBEDDINGS COMPLETE")
    print(f"{'='*70}")
    print(f"Entries embedded: {n_entries}")
    print(f"Models with embeddings: {len(dict_embeddings)}")
    for model_name, emb in dict_embeddings.items():
        print(f"  - {model_name:20s}: {emb.shape}")
    print(f"\nContextual embedding approach:")
    print(f"  - Each term embedded WITH topic context")
    print(f"  - Same term in different topics gets different embeddings")
    print(f"  - Embeddings align 1:1 with df_dict rows")
    print(f"{'='*70}")

elif VIZ_AVAILABLE:
    print("\n Skipping dictionary embeddings - no embedding models loaded")
    dict_embeddings = {}
else:
    print(" Skipping dictionary embeddings - VIZ_AVAILABLE = False")



GENERATING DICTIONARY TERM EMBEDDINGS

Dictionary entries to embed: 782
Format: 'Topic: term' (provides context)
Example: 'Contemporary_Manifestations: aansluit'

pretrained_bertje:
  Generating embeddings for 782 entries...


Embedding:   0%|          | 0/25 [00:00<?, ?it/s]

  Generated: (782, 768)
  Shape: [782 entries x 768 dimensions]
  Each term embedded WITH its topic context

slavery_trained:
  Generating embeddings for 782 entries...


Embedding:   0%|          | 0/25 [00:00<?, ?it/s]

  Generated: (782, 768)
  Shape: [782 entries x 768 dimensions]
  Each term embedded WITH its topic context

DICTIONARY EMBEDDINGS COMPLETE
Entries embedded: 782
Models with embeddings: 2
  - pretrained_bertje   : (782, 768)
  - slavery_trained     : (782, 768)

Contextual embedding approach:
  - Each term embedded WITH topic context
  - Same term in different topics gets different embeddings
  - Embeddings align 1:1 with df_dict rows


In [16]:
# ============================================================
# CELL 9.6: GENERATE CHUNK EMBEDDINGS (SAMPLE)
# ============================================================

if VIZ_AVAILABLE and len([m for m in models.keys() if m != 'base_cosine']) > 0:
    print(f"\n{'='*70}")
    print("GENERATING CHUNK EMBEDDINGS (SAMPLED)")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Sample Chunks (Stratified by Primary Topic)
    # --------------------------------------------------------

    print(f"\n1️⃣  Sampling chunks...")
    print(f"   Total chunks available: {len(df_cosine)}")
    print(f"   Max sample size (SAMPLE_SIZE_3D): {SAMPLE_SIZE_3D}")

    if len(df_cosine) <= SAMPLE_SIZE_3D:
        # Use all chunks
        df_chunks_sampled = df_cosine.copy()
        print(f"   Using all {len(df_chunks_sampled)} chunks (below sample limit)")
    else:
        # Stratified sampling by primary topic
        print(f"   Performing stratified sampling by primary topic...")

        # Determine primary topic for each chunk (highest score)
        primary_topic_idx = df_cosine[score_cols].values.argmax(axis=1)
        df_cosine['primary_topic_temp'] = [topics[i] for i in primary_topic_idx]

        # Calculate samples per topic (proportional to topic frequency)
        topic_counts = df_cosine['primary_topic_temp'].value_counts()
        samples_per_topic = {}

        for topic, count in topic_counts.items():
            proportion = count / len(df_cosine)
            n_samples = max(1, int(proportion * SAMPLE_SIZE_3D))  # At least 1 per topic
            samples_per_topic[topic] = min(n_samples, count)  # Don't oversample

        # Sample from each topic
        sampled_dfs = []
        for topic, n_samples in samples_per_topic.items():
            topic_df = df_cosine[df_cosine['primary_topic_temp'] == topic]
            sampled = topic_df.sample(n=n_samples, random_state=PCA_RANDOM_STATE)
            sampled_dfs.append(sampled)

        df_chunks_sampled = pd.concat(sampled_dfs, ignore_index=True)

        print(f"   ✓ Sampled {len(df_chunks_sampled)} chunks")
        print(f"\n   Sample distribution by topic:")
        for topic in topics:
            count = (df_chunks_sampled['primary_topic_temp'] == topic).sum()
            pct = count / len(df_chunks_sampled) * 100
            print(f"     {topic:50s}: {count:4d} ({pct:5.1f}%)")

    # --------------------------------------------------------
    # 2. Extract Text for Embedding
    # --------------------------------------------------------

    print(f"\n2️⃣  Extracting text for embedding...")

    # Use text_col identified in Cell 9.3
    if text_col and text_col in df_chunks_sampled.columns:
        chunk_texts = df_chunks_sampled[text_col].fillna('').tolist()
        print(f"   ✓ Using column: {text_col}")
        print(f"   ✓ Extracted {len(chunk_texts)} texts")
    else:
        print(f"   ❌ ERROR: Text column '{text_col}' not found!")
        VIZ_AVAILABLE = False

    # --------------------------------------------------------
    # 3. Generate Embeddings for Each Model
    # --------------------------------------------------------

    if VIZ_AVAILABLE:
        chunk_embeddings = {}

        for model_name in ['pretrained_bertje', 'slavery_trained', 'policy_trained']:
            if COMPARE_MODELS.get(model_name, False) and model_name in models:
                print(f"\n3️⃣  {model_name}:")
                print(f"   Generating embeddings for {len(chunk_texts)} chunks...")

                try:
                    embeddings = generate_embeddings(
                        chunk_texts,
                        models[model_name],
                        tokenizers[model_name],
                        batch_size=16,  # Smaller batch for longer texts
                        max_length=512   # Full context for chunks
                    )

                    chunk_embeddings[model_name] = embeddings

                    print(f"   ✓ Generated: {embeddings.shape}")
                    print(f"     Shape: [{len(chunk_texts)} chunks × {embeddings.shape[1]} dimensions]")

                except Exception as e:
                    print(f"   ❌ Failed: {e}")
                    # Don't add to chunk_embeddings

        # --------------------------------------------------------
        # Summary
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("CHUNK EMBEDDINGS COMPLETE")
        print(f"{'='*70}")
        print(f"✓ Chunks embedded: {len(df_chunks_sampled)}")
        print(f"✓ Models with embeddings: {len(chunk_embeddings)}")
        for model_name, emb in chunk_embeddings.items():
            print(f"  - {model_name:20s}: {emb.shape}")
        print(f"{'='*70}")
        print(f"\n✅ Section 2 Complete - Ready for visualizations!")

elif VIZ_AVAILABLE:
    print("\n⊘ Skipping chunk embeddings - no embedding models loaded")
    chunk_embeddings = {}
    df_chunks_sampled = pd.DataFrame()
else:
    print("⚠ Skipping chunk embeddings - VIZ_AVAILABLE = False")


GENERATING CHUNK EMBEDDINGS (SAMPLED)

1️⃣  Sampling chunks...
   Total chunks available: 2840
   Max sample size (SAMPLE_SIZE_3D): 1000
   Performing stratified sampling by primary topic...
   ✓ Sampled 999 chunks

   Sample distribution by topic:
     Contemporary_Manifestations                       :  335 ( 33.5%)
     Historical_Slavery_Colonialism                    :  485 ( 48.5%)
     Structural_Continuity_Neocolonial                 :  179 ( 17.9%)

2️⃣  Extracting text for embedding...
   ✓ Using column: text_for_scoring
   ✓ Extracted 999 texts

3️⃣  pretrained_bertje:
   Generating embeddings for 999 chunks...


Embedding:   0%|          | 0/63 [00:00<?, ?it/s]

   ✓ Generated: (999, 768)
     Shape: [999 chunks × 768 dimensions]

3️⃣  slavery_trained:
   Generating embeddings for 999 chunks...


Embedding:   0%|          | 0/63 [00:00<?, ?it/s]

   ✓ Generated: (999, 768)
     Shape: [999 chunks × 768 dimensions]

CHUNK EMBEDDINGS COMPLETE
✓ Chunks embedded: 999
✓ Models with embeddings: 2
  - pretrained_bertje   : (999, 768)
  - slavery_trained     : (999, 768)

✅ Section 2 Complete - Ready for visualizations!


---
## SECTION 3: Dictionary Fitness Visualizations
---

**Goal**: Validate that the V10 dictionary is well-constructed and fit for purpose.

**Key Questions**:
1. Do weight tiers reflect semantic quality? (Higher weights = more topic-specific terms?)
2. Did expansion improve dictionary coverage without noise?
3. Do dictionary terms cluster by topic across different models?
4. How do models differ in their representation of dictionary terms?

**Visualizations**:
- **9.7**: Weight Tier Validation (Boxplot)
- **9.8**: Expansion Quality Validation (2D Scatter)
- **9.9**: Dictionary Term Clustering - 2D Multi-Model Comparison
- **9.10**: Dictionary Term Clustering - 3D Exploration

In [18]:
# ============================================================
# CELL 9.7: WEIGHT TIER VALIDATION (Multi-Model) - FIXED
# ============================================================

if VIZ_AVAILABLE and len(dict_embeddings) > 0:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.7: Weight Tier Validation")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Calculate Intra-Topic Distances for All Models
    # --------------------------------------------------------

    print(f"\n1. Calculating intra-topic distances for all models...")

    models_to_use = list(dict_embeddings.keys())
    print(f"   Models: {', '.join(models_to_use)}")

    # Store distances per model
    model_distances = {}

    for model_name in models_to_use:
        print(f"\n   Processing {model_name}...")

        term_embeddings = dict_embeddings[model_name]

        # Calculate intra-topic distance for each term
        df_dict[f'intra_topic_distance_{model_name}'] = np.nan

        for topic in topics:
            topic_mask = df_dict['topic'] == topic
            topic_indices = df_dict[topic_mask].index.tolist()

            if len(topic_indices) < 2:
                continue

            # Get embeddings for this topic's terms
            topic_term_embeddings = term_embeddings[topic_mask]

            # Calculate pairwise cosine similarities
            similarities = cosine_similarity(topic_term_embeddings)

            # For each term, average similarity to OTHER terms (exclude self)
            for i, idx in enumerate(topic_indices):
                # Exclude diagonal (self-similarity = 1.0)
                other_sims = np.concatenate([similarities[i, :i], similarities[i, i+1:]])
                avg_similarity = other_sims.mean()

                # Convert to distance (1 - similarity)
                df_dict.loc[idx, f'intra_topic_distance_{model_name}'] = 1 - avg_similarity

        print(f"     Calculated distances for {len(df_dict)} terms")

    # --------------------------------------------------------
    # 2. Create Weight Tier Categories from Dictionary
    # --------------------------------------------------------

    print(f"\n2. Identifying weight tiers from dictionary...")

    if 'weight' in df_dict.columns and 'category' in df_dict.columns:
        # Use the category column directly as the tier label
        df_dict['weight_tier'] = df_dict['category']

        # Get unique categories and their corresponding average weights
        category_weights = df_dict.groupby('category')['weight'].mean().sort_values(ascending=False)

        # Create ordered list from highest weight to lowest
        tier_order = category_weights.index.tolist()

        print(f"   Found {len(tier_order)} weight categories (ordered by average weight):")
        print(f"\n   Distribution:")
        for tier in tier_order:
            count = (df_dict['weight_tier'] == tier).sum()
            avg_weight = category_weights[tier]
            weight_range = df_dict[df_dict['category'] == tier]['weight']
            min_weight = weight_range.min()
            max_weight = weight_range.max()
            print(f"     {tier:30s}: {count:3d} terms | Avg weight: {avg_weight:.3f} | Range: [{min_weight:.3f}, {max_weight:.3f}]")

    elif 'weight' in df_dict.columns:
        # Fallback: If category column doesn't exist, create basic tiers from weight
        print("   WARNING: 'category' column not found, creating tiers from weight values")

        unique_weights = sorted(df_dict['weight'].unique(), reverse=True)

        # Group similar weights into tiers
        df_dict['weight_tier'] = df_dict['weight'].apply(lambda w: f"Weight: {w:.2f}")
        tier_order = [f"Weight: {w:.2f}" for w in unique_weights]

        print(f"   Created {len(tier_order)} weight tiers")

    else:
        print("   ERROR: 'weight' column not found in dictionary")
        print("   Cannot perform weight tier validation")
        tier_order = []

    # --------------------------------------------------------
    # 3. Create Combined Statistical Table
    # --------------------------------------------------------

    if len(tier_order) > 0:
        print(f"\n3. Creating combined statistics table...")

        stats_data = []

        for model_name in models_to_use:
            distance_col = f'intra_topic_distance_{model_name}'
            df_plot = df_dict[df_dict[distance_col].notna()].copy()

            for tier in tier_order:
                tier_data = df_plot[df_plot['weight_tier'] == tier][distance_col]
                if len(tier_data) == 0:
                    continue

                stats_data.append({
                    'Model': model_name,
                    'Weight_Tier': tier,
                    'Mean_Distance': tier_data.mean(),
                    'Std_Distance': tier_data.std(),
                    'Median_Distance': tier_data.median(),
                    'N_Terms': len(tier_data)
                })

        df_stats = pd.DataFrame(stats_data)

        # Save combined stats table
        stats_path = visuals_dir / 'weight_tier_stats_combined.csv'
        df_stats.to_csv(stats_path, index=False)
        print(f"   Saved: {stats_path.name}")

        # --------------------------------------------------------
        # 4. Create Separate Visualizations Per Model
        # --------------------------------------------------------

        print(f"\n4. Creating visualizations...")

        for model_name in models_to_use:
            print(f"\n   Creating visualization for {model_name}...")

            distance_col = f'intra_topic_distance_{model_name}'
            df_plot = df_dict[df_dict[distance_col].notna()].copy()

            fig = go.Figure()

            # Add boxplot for each tier
            for tier in tier_order:
                tier_data = df_plot[df_plot['weight_tier'] == tier]

                if len(tier_data) == 0:
                    continue

                fig.add_trace(go.Box(
                    y=tier_data[distance_col],
                    name=tier,
                    boxmean='sd',  # Show mean and std dev
                    marker_color='lightblue',
                    hovertemplate=(
                        '<b>%{y:.3f}</b><br>' +
                        'Tier: ' + tier + '<br>' +
                        '<extra></extra>'
                    )
                ))

            # Update layout
            fig.update_layout(
                title={
                    'text': f"Weight Tier Validation: {model_name}<br><sub>Intra-Topic Distance by Weight Tier | Lower distance = more topic-coherent</sub>",
                    'x': 0.5,
                    'xanchor': 'center'
                },
                xaxis_title="Weight Tier (from dictionary category column)",
                yaxis_title="Intra-Topic Distance (1 - cosine similarity)",
                height=600,
                showlegend=False,
                hovermode='closest',
                template='plotly_white'
            )

            # Add annotation explaining expectation
            fig.add_annotation(
                text="Expected: Higher weight tiers should have LOWER distances (more coherent)",
                xref="paper", yref="paper",
                x=0.5, y=-0.15,
                showarrow=False,
                font=dict(size=10, color="gray"),
                xanchor='center'
            )

            # Display
            if SHOW_IN_NOTEBOOK:
                fig.show()

            # Save interactive
            if SAVE_INTERACTIVE:
                output_path = visuals_dir / f'weight_tier_validation_{model_name}.html'
                fig.write_html(str(output_path))
                print(f"     Saved interactive: {output_path.name}")

            # Save static
            if SAVE_STATIC:
                output_path = visuals_dir / f'weight_tier_validation_{model_name}.png'
                fig.write_image(str(output_path), width=1200, height=800, scale=2)
                print(f"     Saved static: {output_path.name}")

        # --------------------------------------------------------
        # 5. Statistical Summary
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("WEIGHT TIER VALIDATION SUMMARY")
        print(f"{'='*70}")

        for model_name in models_to_use:
            print(f"\n{model_name}:")
            model_stats = df_stats[df_stats['Model'] == model_name]

            for _, row in model_stats.iterrows():
                tier = row['Weight_Tier']
                print(f"  {tier}:")
                print(f"    Mean distance: {row['Mean_Distance']:.3f}")
                print(f"    Std dev:       {row['Std_Distance']:.3f}")
                print(f"    Median:        {row['Median_Distance']:.3f}")
                print(f"    N terms:       {int(row['N_Terms'])}")

        print(f"\n{'='*70}")

    else:
        print("\n   Skipping visualization - no valid weight tiers found")

else:
    print("\n Skipping weight tier validation - no embeddings available")


Initialized section3_figs list for storing visualizations


In [21]:
# ============================================================
# CELL 9.7: WEIGHT TIER VALIDATION (Multi-Model) - FIXED
# ============================================================

if VIZ_AVAILABLE and len(dict_embeddings) > 0:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.7: Weight Tier Validation")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Calculate Intra-Topic Distances for All Models
    # --------------------------------------------------------

    print(f"\n1. Calculating intra-topic distances for all models...")

    models_to_use = list(dict_embeddings.keys())
    print(f"   Models: {', '.join(models_to_use)}")

    # Store distances per model
    model_distances = {}

    for model_name in models_to_use:
        print(f"\n   Processing {model_name}...")

        term_embeddings = dict_embeddings[model_name]

        # Calculate intra-topic distance for each term
        df_dict[f'intra_topic_distance_{model_name}'] = np.nan

        for topic in topics:
            topic_mask = df_dict['topic'] == topic
            topic_indices = df_dict[topic_mask].index.tolist()

            if len(topic_indices) < 2:
                continue

            # Get embeddings for this topic's terms
            topic_term_embeddings = term_embeddings[topic_mask]

            # Calculate pairwise cosine similarities
            similarities = cosine_similarity(topic_term_embeddings)

            # For each term, average similarity to OTHER terms (exclude self)
            for i, idx in enumerate(topic_indices):
                # Exclude diagonal (self-similarity = 1.0)
                other_sims = np.concatenate([similarities[i, :i], similarities[i, i+1:]])
                avg_similarity = other_sims.mean()

                # Convert to distance (1 - similarity)
                df_dict.loc[idx, f'intra_topic_distance_{model_name}'] = 1 - avg_similarity

        print(f"     Calculated distances for {len(df_dict)} terms")

    # --------------------------------------------------------
    # 2. Create Weight Tier Categories from Dictionary
    # --------------------------------------------------------

    print(f"\n2. Identifying weight tiers from dictionary...")

    if 'weight' in df_dict.columns and 'category' in df_dict.columns:
        # Use the category column directly as the tier label
        df_dict['weight_tier'] = df_dict['category']

        # Get unique categories and their corresponding average weights
        category_weights = df_dict.groupby('category')['weight'].mean().sort_values(ascending=False)

        # Create ordered list from highest weight to lowest
        tier_order = category_weights.index.tolist()

        print(f"   Found {len(tier_order)} weight categories (ordered by average weight):")
        print(f"\n   Distribution:")
        for tier in tier_order:
            count = (df_dict['weight_tier'] == tier).sum()
            avg_weight = category_weights[tier]
            weight_range = df_dict[df_dict['category'] == tier]['weight']
            min_weight = weight_range.min()
            max_weight = weight_range.max()
            print(f"     {tier:30s}: {count:3d} terms | Avg weight: {avg_weight:.3f} | Range: [{min_weight:.3f}, {max_weight:.3f}]")

    elif 'weight' in df_dict.columns:
        # Fallback: If category column doesn't exist, create basic tiers from weight
        print("   WARNING: 'category' column not found, creating tiers from weight values")

        unique_weights = sorted(df_dict['weight'].unique(), reverse=True)

        # Group similar weights into tiers
        df_dict['weight_tier'] = df_dict['weight'].apply(lambda w: f"Weight: {w:.2f}")
        tier_order = [f"Weight: {w:.2f}" for w in unique_weights]

        print(f"   Created {len(tier_order)} weight tiers")

    else:
        print("   ERROR: 'weight' column not found in dictionary")
        print("   Cannot perform weight tier validation")
        tier_order = []

    # --------------------------------------------------------
    # 3. Create Combined Statistical Table
    # --------------------------------------------------------

    if len(tier_order) > 0:
        print(f"\n3. Creating combined statistics table...")

        stats_data = []

        for model_name in models_to_use:
            distance_col = f'intra_topic_distance_{model_name}'
            df_plot = df_dict[df_dict[distance_col].notna()].copy()

            for tier in tier_order:
                tier_data = df_plot[df_plot['weight_tier'] == tier][distance_col]
                if len(tier_data) == 0:
                    continue

                stats_data.append({
                    'Model': model_name,
                    'Weight_Tier': tier,
                    'Mean_Distance': tier_data.mean(),
                    'Std_Distance': tier_data.std(),
                    'Median_Distance': tier_data.median(),
                    'N_Terms': len(tier_data)
                })

        df_stats = pd.DataFrame(stats_data)

        # Save combined stats table
        stats_path = visuals_dir / 'weight_tier_stats_combined.csv'
        df_stats.to_csv(stats_path, index=False)
        print(f"   Saved: {stats_path.name}")

        # --------------------------------------------------------
        # 4. Create Separate Visualizations Per Model
        # --------------------------------------------------------

        print(f"\n4. Creating visualizations...")

        for model_name in models_to_use:
            print(f"\n   Creating visualization for {model_name}...")

            distance_col = f'intra_topic_distance_{model_name}'
            df_plot = df_dict[df_dict[distance_col].notna()].copy()

            fig = go.Figure()

            # Add boxplot for each tier
            for tier in tier_order:
                tier_data = df_plot[df_plot['weight_tier'] == tier]

                if len(tier_data) == 0:
                    continue

                fig.add_trace(go.Box(
                    y=tier_data[distance_col],
                    name=tier,
                    boxmean='sd',  # Show mean and std dev
                    marker_color='lightblue',
                    hovertemplate=(
                        '<b>%{y:.3f}</b><br>' +
                        'Tier: ' + tier + '<br>' +
                        '<extra></extra>'
                    )
                ))

            # Update layout
            fig.update_layout(
                title={
                    'text': f"Weight Tier Validation: {model_name}<br><sub>Intra-Topic Distance by Weight Tier | Lower distance = more topic-coherent</sub>",
                    'x': 0.5,
                    'xanchor': 'center'
                },
                xaxis_title="Weight Tier (from dictionary category column)",
                yaxis_title="Intra-Topic Distance (1 - cosine similarity)",
                height=600,
                showlegend=False,
                hovermode='closest',
                template='plotly_white'
            )

            # Add annotation explaining expectation
            fig.add_annotation(
                text="Expected: Higher weight tiers should have LOWER distances (more coherent)",
                xref="paper", yref="paper",
                x=0.5, y=-0.15,
                showarrow=False,
                font=dict(size=10, color="gray"),
                xanchor='center'
            )

            # Display
            if SHOW_IN_NOTEBOOK:
                fig.show()

            # Save interactive
            if SAVE_INTERACTIVE:
                output_path = visuals_dir / f'weight_tier_validation_{model_name}.html'
                fig.write_html(str(output_path))
                print(f"     Saved interactive: {output_path.name}")

            # Save static
            if SAVE_STATIC:
                output_path = visuals_dir / f'weight_tier_validation_{model_name}.png'
                fig.write_image(str(output_path), width=1200, height=800, scale=2)
                print(f"     Saved static: {output_path.name}")

        # --------------------------------------------------------
        # 5. Statistical Summary
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("WEIGHT TIER VALIDATION SUMMARY")
        print(f"{'='*70}")

        for model_name in models_to_use:
            print(f"\n{model_name}:")
            model_stats = df_stats[df_stats['Model'] == model_name]

            for _, row in model_stats.iterrows():
                tier = row['Weight_Tier']
                print(f"  {tier}:")
                print(f"    Mean distance: {row['Mean_Distance']:.3f}")
                print(f"    Std dev:       {row['Std_Distance']:.3f}")
                print(f"    Median:        {row['Median_Distance']:.3f}")
                print(f"    N terms:       {int(row['N_Terms'])}")

        print(f"\n{'='*70}")

    else:
        print("\n   Skipping visualization - no valid weight tiers found")

else:
    print("\n Skipping weight tier validation - no embeddings available")



VISUALIZATION 9.7: Weight Tier Validation

1. Calculating intra-topic distances for all models...
   Models: pretrained_bertje, slavery_trained

   Processing pretrained_bertje...
     Calculated distances for 782 terms

   Processing slavery_trained...
     Calculated distances for 782 terms

2. Identifying weight tiers from dictionary...
   Found 5 weight categories (ordered by average weight):

   Distribution:
     core_problem                  : 173 terms | Avg weight: 1.000 | Range: [1.000, 1.000]
     strong_problem                : 461 terms | Avg weight: 0.950 | Range: [0.950, 0.950]
     related_strong                :  48 terms | Avg weight: 0.850 | Range: [0.850, 0.850]
     related_moderate              :  79 terms | Avg weight: 0.750 | Range: [0.750, 0.750]
     era_context                   :  21 terms | Avg weight: 0.550 | Range: [0.550, 0.550]

3. Creating combined statistics table...
   Saved: weight_tier_stats_combined.csv

4. Creating visualizations...

   Creating

     Saved interactive: weight_tier_validation_pretrained_bertje.html

   Creating visualization for slavery_trained...


     Saved interactive: weight_tier_validation_slavery_trained.html

WEIGHT TIER VALIDATION SUMMARY

pretrained_bertje:
  core_problem:
    Mean distance: 0.058
    Std dev:       0.016
    Median:        0.054
    N terms:       173
  strong_problem:
    Mean distance: 0.055
    Std dev:       0.021
    Median:        0.049
    N terms:       461
  related_strong:
    Mean distance: 0.056
    Std dev:       0.014
    Median:        0.052
    N terms:       48
  related_moderate:
    Mean distance: 0.056
    Std dev:       0.020
    Median:        0.052
    N terms:       79
  era_context:
    Mean distance: 0.067
    Std dev:       0.026
    Median:        0.058
    N terms:       21

slavery_trained:
  core_problem:
    Mean distance: 0.268
    Std dev:       0.074
    Median:        0.268
    N terms:       173
  strong_problem:
    Mean distance: 0.243
    Std dev:       0.103
    Median:        0.217
    N terms:       461
  related_strong:
    Mean distance: 0.261
    Std dev:   

In [23]:
# ============================================================
# CELL 9.8: EXPANSION QUALITY VALIDATION (Multi-Model)
# ============================================================

if VIZ_AVAILABLE and len(dict_embeddings) > 0:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.8: Expansion Quality Validation")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Identify Seeds vs. Expanded Terms
    # --------------------------------------------------------

    print(f"\n1. Identifying seed vs. expanded terms...")

    if 'is_seed' in df_dict.columns:
        seed_mask = df_dict['is_seed'] == True
        expanded_mask = df_dict['is_seed'] == False
    elif 'category' in df_dict.columns:
        # Fallback: assume specific categories are seeds
        seed_categories = ['KERN', 'BELEID', 'STERK']
        seed_mask = df_dict['category'].isin(seed_categories)
        expanded_mask = ~seed_mask
    else:
        print("   WARNING: Cannot identify seeds - no 'is_seed' or 'category' column")
        print("   Skipping expansion validation")
        VIZ_AVAILABLE = False

    if VIZ_AVAILABLE:
        n_seeds = seed_mask.sum()
        n_expanded = expanded_mask.sum()

        print(f"   Seeds:    {n_seeds}")
        print(f"   Expanded: {n_expanded}")

        # --------------------------------------------------------
        # 2. Calculate Per-Model Centroids and Distances
        # --------------------------------------------------------

        print(f"\n2. Processing all models...")

        models_to_use = list(dict_embeddings.keys())
        print(f"   Models: {', '.join(models_to_use)}")

        model_results = {}

        for model_name in models_to_use:
            print(f"\n   Processing {model_name}...")

            term_embeddings = dict_embeddings[model_name]

            # Calculate centroid for each topic (mean of seed embeddings)
            topic_centroids = {}

            for topic in topics:
                topic_seeds_mask = seed_mask & (df_dict['topic'] == topic)

                if topic_seeds_mask.sum() == 0:
                    continue

                seed_embeddings = term_embeddings[topic_seeds_mask]
                centroid = seed_embeddings.mean(axis=0)
                topic_centroids[topic] = centroid

            print(f"     Calculated centroids for {len(topic_centroids)} topics")

            # Calculate distance to centroid for all terms
            df_dict[f'distance_to_centroid_{model_name}'] = np.nan

            for topic in topics:
                if topic not in topic_centroids:
                    continue

                topic_mask = df_dict['topic'] == topic
                topic_embeddings = term_embeddings[topic_mask]
                centroid = topic_centroids[topic]

                # Cosine distance to centroid
                similarities = cosine_similarity(topic_embeddings, centroid.reshape(1, -1))
                distances = 1 - similarities.flatten()

                df_dict.loc[topic_mask, f'distance_to_centroid_{model_name}'] = distances

            print(f"     Calculated distances for {len(df_dict)} terms")

            # PCA for 2D visualization
            pca = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
            term_embeddings_2d = pca.fit_transform(term_embeddings)

            # Also project centroids to 2D
            if len(topic_centroids) > 0:
                centroid_matrix = np.array([topic_centroids[t] for t in sorted(topic_centroids.keys())])
                centroids_2d = pca.transform(centroid_matrix)
                centroid_coords = {topic: centroids_2d[i] for i, topic in enumerate(sorted(topic_centroids.keys()))}
            else:
                centroid_coords = {}

            print(f"     PCA complete: {term_embeddings_2d.shape}")

            # Store results
            model_results[model_name] = {
                'embeddings_2d': term_embeddings_2d,
                'centroids': topic_centroids,
                'centroid_coords': centroid_coords,
                'pca': pca,
                'pca_variance': pca.explained_variance_ratio_
            }

        # --------------------------------------------------------
        # 3. Create Combined Statistics Table
        # --------------------------------------------------------

        print(f"\n3. Creating combined statistics table...")

        stats_data = []

        for model_name in models_to_use:
            distance_col = f'distance_to_centroid_{model_name}'

            # Seeds
            seed_distances = df_dict[seed_mask][distance_col].dropna()
            if len(seed_distances) > 0:
                stats_data.append({
                    'Model': model_name,
                    'Group': 'Seeds',
                    'Mean_Distance': seed_distances.mean(),
                    'Std_Distance': seed_distances.std(),
                    'Median_Distance': seed_distances.median(),
                    'Min_Distance': seed_distances.min(),
                    'Max_Distance': seed_distances.max(),
                    'N_Terms': len(seed_distances)
                })

            # Expanded
            expanded_distances = df_dict[expanded_mask][distance_col].dropna()
            if len(expanded_distances) > 0:
                stats_data.append({
                    'Model': model_name,
                    'Group': 'Expanded',
                    'Mean_Distance': expanded_distances.mean(),
                    'Std_Distance': expanded_distances.std(),
                    'Median_Distance': expanded_distances.median(),
                    'Min_Distance': expanded_distances.min(),
                    'Max_Distance': expanded_distances.max(),
                    'N_Terms': len(expanded_distances)
                })

            # Calculate separation (expanded - seeds mean distance)
            if len(seed_distances) > 0 and len(expanded_distances) > 0:
                separation = expanded_distances.mean() - seed_distances.mean()
                stats_data.append({
                    'Model': model_name,
                    'Group': 'Separation',
                    'Mean_Distance': separation,
                    'Std_Distance': np.nan,
                    'Median_Distance': np.nan,
                    'Min_Distance': np.nan,
                    'Max_Distance': np.nan,
                    'N_Terms': np.nan
                })

        df_stats = pd.DataFrame(stats_data)

        # Save combined stats table
        stats_path = visuals_dir / 'expansion_quality_stats_combined.csv'
        df_stats.to_csv(stats_path, index=False)
        print(f"   Saved: {stats_path.name}")

        # --------------------------------------------------------
        # 4. Create Separate Visualizations Per Model
        # --------------------------------------------------------

        print(f"\n4. Creating visualizations...")

        for model_name in models_to_use:
            print(f"\n   Creating visualization for {model_name}...")

            # Get model-specific data
            embeddings_2d = model_results[model_name]['embeddings_2d']
            centroid_coords = model_results[model_name]['centroid_coords']
            pca_variance = model_results[model_name]['pca_variance']

            # Add to dataframe
            df_dict[f'pca_x_{model_name}'] = embeddings_2d[:, 0]
            df_dict[f'pca_y_{model_name}'] = embeddings_2d[:, 1]

            fig = go.Figure()

            # Plot by topic
            for topic in topics:
                topic_mask = df_dict['topic'] == topic

                # Seeds for this topic
                topic_seeds = df_dict[topic_mask & seed_mask]
                if len(topic_seeds) > 0:
                    fig.add_trace(go.Scatter(
                        x=topic_seeds[f'pca_x_{model_name}'],
                        y=topic_seeds[f'pca_y_{model_name}'],
                        mode='markers',
                        name=f'{topic} (Seeds)',
                        marker=dict(size=10, symbol='circle', line=dict(width=2, color='black')),
                        hovertemplate=(
                            '<b>%{text}</b><br>' +
                            f'Topic: {topic}<br>' +
                            'Group: Seed<br>' +
                            f'Distance: %{{customdata:.3f}}<br>' +
                            '<extra></extra>'
                        ),
                        text=topic_seeds['term'],
                        customdata=topic_seeds[f'distance_to_centroid_{model_name}'],
                        legendgroup=topic
                    ))

                # Expanded for this topic
                topic_expanded = df_dict[topic_mask & expanded_mask]
                if len(topic_expanded) > 0:
                    fig.add_trace(go.Scatter(
                        x=topic_expanded[f'pca_x_{model_name}'],
                        y=topic_expanded[f'pca_y_{model_name}'],
                        mode='markers',
                        name=f'{topic} (Expanded)',
                        marker=dict(size=8, symbol='circle-open'),
                        hovertemplate=(
                            '<b>%{text}</b><br>' +
                            f'Topic: {topic}<br>' +
                            'Group: Expanded<br>' +
                            f'Distance: %{{customdata:.3f}}<br>' +
                            '<extra></extra>'
                        ),
                        text=topic_expanded['term'],
                        customdata=topic_expanded[f'distance_to_centroid_{model_name}'],
                        legendgroup=topic,
                        showlegend=True
                    ))

                # Centroid
                if topic in centroid_coords:
                    coords = centroid_coords[topic]
                    fig.add_trace(go.Scatter(
                        x=[coords[0]],
                        y=[coords[1]],
                        mode='markers',
                        name=f'{topic} (Centroid)',
                        marker=dict(size=15, symbol='star', color='red'),
                        hovertemplate=(
                            f'<b>Centroid: {topic}</b><br>' +
                            '<extra></extra>'
                        ),
                        legendgroup=topic,
                        showlegend=False
                    ))

            # Update layout
            fig.update_layout(
                title={
                    'text': f"Expansion Quality: {model_name}<br><sub>2D PCA Projection | Seeds (solid) vs Expanded (hollow) | Red stars = Topic centroids</sub>",
                    'x': 0.5,
                    'xanchor': 'center'
                },
                xaxis_title=f"PC1 ({pca_variance[0]:.1%} variance)",
                yaxis_title=f"PC2 ({pca_variance[1]:.1%} variance)",
                height=700,
                hovermode='closest',
                template='plotly_white',
                legend=dict(
                    title="Topic & Group",
                    orientation="v",
                    yanchor="top",
                    y=1,
                    xanchor="left",
                    x=1.02
                )
            )

            # Add annotation
            fig.add_annotation(
                text="Expected: Seeds (solid) cluster tightly around centroids (stars)<br>Expanded (hollow) should be nearby but may be more dispersed",
                xref="paper", yref="paper",
                x=0.5, y=-0.15,
                showarrow=False,
                font=dict(size=10, color="gray"),
                xanchor='center'
            )

            # Display
            if SHOW_IN_NOTEBOOK:
                section3_figs.append(fig)  # Store for export
                fig.show()

            # Save interactive
            if SAVE_INTERACTIVE:
                output_path = visuals_dir / f'expansion_quality_{model_name}.html'
                fig.write_html(str(output_path))
                print(f"     Saved interactive: {output_path.name}")

            # Save static
            if SAVE_STATIC:
                output_path = visuals_dir / f'expansion_quality_{model_name}.png'
                fig.write_image(str(output_path), width=1400, height=900, scale=2)
                print(f"     Saved static: {output_path.name}")

        # --------------------------------------------------------
        # 5. Statistical Summary
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("EXPANSION QUALITY SUMMARY")
        print(f"{'='*70}")

        for model_name in models_to_use:
            print(f"\n{model_name}:")
            model_stats = df_stats[df_stats['Model'] == model_name]

            for _, row in model_stats.iterrows():
                group = row['Group']
                if group == 'Separation':
                    print(f"  {group}:")
                    print(f"    Expanded - Seeds: {row['Mean_Distance']:.3f}")
                    print(f"    (Positive = expanded terms farther from centroids)")
                else:
                    print(f"  {group}:")
                    print(f"    Mean distance:   {row['Mean_Distance']:.3f}")
                    print(f"    Std dev:         {row['Std_Distance']:.3f}")
                    print(f"    Median:          {row['Median_Distance']:.3f}")
                    print(f"    N terms:         {int(row['N_Terms'])}")

        print(f"\n{'='*70}")
        print("INTERPRETATION")
        print(f"{'='*70}")
        print("Good expansion quality:")
        print("  - Seeds have LOW mean distance (tight cluster)")
        print("  - Expanded terms have MODERATE distance (nearby but exploring)")
        print("  - Positive separation (expanded > seeds) indicates semantic expansion")
        print("  - Small separation suggests conservative expansion (stays close)")
        print(f"{'='*70}")

else:
    print("\n Skipping expansion quality validation - no embeddings available")



VISUALIZATION 9.8: Expansion Quality Validation

1. Identifying seed vs. expanded terms...
   Seeds:    143
   Expanded: 639

2. Processing all models...
   Models: pretrained_bertje, slavery_trained

   Processing pretrained_bertje...
     Calculated centroids for 3 topics
     Calculated distances for 782 terms
     PCA complete: (782, 2)

   Processing slavery_trained...
     Calculated centroids for 3 topics
     Calculated distances for 782 terms
     PCA complete: (782, 2)

3. Creating combined statistics table...
   Saved: expansion_quality_stats_combined.csv

4. Creating visualizations...

   Creating visualization for pretrained_bertje...


     Saved interactive: expansion_quality_pretrained_bertje.html

   Creating visualization for slavery_trained...


     Saved interactive: expansion_quality_slavery_trained.html

EXPANSION QUALITY SUMMARY

pretrained_bertje:
  Seeds:
    Mean distance:   0.029
    Std dev:         0.013
    Median:          0.027
    N terms:         143
  Expanded:
    Mean distance:   0.030
    Std dev:         0.015
    Median:          0.026
    N terms:         639
  Separation:
    Expanded - Seeds: 0.001
    (Positive = expanded terms farther from centroids)

slavery_trained:
  Seeds:
    Mean distance:   0.128
    Std dev:         0.075
    Median:          0.111
    N terms:         143
  Expanded:
    Mean distance:   0.151
    Std dev:         0.088
    Median:          0.129
    N terms:         639
  Separation:
    Expanded - Seeds: 0.024
    (Positive = expanded terms farther from centroids)

INTERPRETATION
Good expansion quality:
  - Seeds have LOW mean distance (tight cluster)
  - Expanded terms have MODERATE distance (nearby but exploring)
  - Positive separation (expanded > seeds) indicates seman

In [24]:
# ============================================================
# CELL 9.9: DICTIONARY TERM CLUSTERING - 2D Multi-Model Comparison
# ============================================================

if VIZ_AVAILABLE and len(dict_embeddings) > 1:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.9: Dictionary Term Clustering - 2D Multi-Model")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Determine Which Models to Compare
    # --------------------------------------------------------

    print(f"\n1️⃣  Models available for comparison:")

    models_to_plot = [m for m in ['pretrained_bertje', 'slavery_trained', 'policy_trained']
                      if m in dict_embeddings]

    for model in models_to_plot:
        print(f"   - {model}")

    if len(models_to_plot) < 2:
        print("\n   ⚠ Need at least 2 models for comparison")
        print("   Skipping multi-model visualization")
    else:
        # --------------------------------------------------------
        # 2. Perform PCA for Each Model
        # --------------------------------------------------------

        print(f"\n2️⃣  Performing PCA for each model...")

        model_pca_results = {}

        for model_name in models_to_plot:
            embeddings = dict_embeddings[model_name]

            pca = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
            embeddings_2d = pca.fit_transform(embeddings)

            model_pca_results[model_name] = {
                'coords': embeddings_2d,
                'var_explained': pca.explained_variance_ratio_
            }

            print(f"   {model_name:20s}: {pca.explained_variance_ratio_.sum():.1%} variance explained")

        # --------------------------------------------------------
        # 3. Create Subplots (One per Model)
        # --------------------------------------------------------

        print(f"\n3️⃣  Creating multi-model visualization...")

        n_models = len(models_to_plot)

        # Create subplot grid
        fig = make_subplots(
            rows=1, cols=n_models,
            subplot_titles=[m.replace('_', ' ').title() for m in models_to_plot],
            horizontal_spacing=0.08
        )

        # Color palette
        colors = px.colors.qualitative.Set2
        if len(topics) > len(colors):
            colors = px.colors.qualitative.Alphabet
        topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

        # Plot each model
        for col_idx, model_name in enumerate(models_to_plot, start=1):
            coords = model_pca_results[model_name]['coords']
            var_explained = model_pca_results[model_name]['var_explained']

            # Add scatter for each topic
            for topic in topics:
                topic_mask = df_dict['topic'] == topic
                topic_coords = coords[topic_mask]
                topic_terms = df_dict[topic_mask]['term'].tolist()

                # Only show legend for first subplot
                show_legend = (col_idx == 1)

                fig.add_trace(
                    go.Scatter(
                        x=topic_coords[:, 0],
                        y=topic_coords[:, 1],
                        mode='markers',
                        name=topic,
                        marker=dict(
                            size=8,
                            color=topic_colors[topic],
                            opacity=0.7
                        ),
                        text=topic_terms,
                        hovertemplate=(
                            '<b>%{text}</b><br>' +
                            f'Topic: {topic}<br>' +
                            f'Model: {model_name}<br>' +
                            '<extra></extra>'
                        ),
                        showlegend=show_legend,
                        legendgroup=topic  # Group legends by topic
                    ),
                    row=1, col=col_idx
                )

            # Update axes labels
            fig.update_xaxes(
                title_text=f"PC1 ({var_explained[0]:.1%})",
                row=1, col=col_idx
            )

            if col_idx == 1:
                fig.update_yaxes(
                    title_text=f"PC2 ({var_explained[1]:.1%})",
                    row=1, col=col_idx
                )

        # Update layout
        fig.update_layout(
            title={
                'text': "Dictionary Term Clustering: Model Comparison (2D PCA)<br><sub>Compare how different models represent dictionary terms in semantic space</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            height=600,
            hovermode='closest',
            template='plotly_white',
            legend=dict(
                orientation="v",
                yanchor="top",
                y=1,
                xanchor="left",
                x=1.02,
                title="Topics"
            )
        )

        # --------------------------------------------------------
        # 4. Display & Save
        # --------------------------------------------------------

        if SHOW_IN_NOTEBOOK:
            section3_figs.append(fig)  # Store for export
            fig.show()

        if SAVE_INTERACTIVE:
            output_path = visuals_dir / 'dict_clustering_2d_multimodel.html'
            fig.write_html(str(output_path))
            print(f"   ✓ Saved interactive: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / 'dict_clustering_2d_multimodel.png'
            fig.write_image(str(output_path), width=1800, height=800, scale=2)
            print(f"   ✓ Saved static: {output_path.name}")

        print(f"\n{'='*70}")
        print("MULTI-MODEL COMPARISON COMPLETE")
        print(f"{'='*70}")

elif VIZ_AVAILABLE and len(dict_embeddings) == 1:
    print("\n⊘ Only 1 embedding model available - skipping multi-model comparison")
else:
    print("\n⊘ Skipping multi-model comparison - no embeddings available")


VISUALIZATION 9.9: Dictionary Term Clustering - 2D Multi-Model

1️⃣  Models available for comparison:
   - pretrained_bertje
   - slavery_trained

2️⃣  Performing PCA for each model...
   pretrained_bertje   : 57.4% variance explained
   slavery_trained     : 44.6% variance explained

3️⃣  Creating multi-model visualization...


   ✓ Saved interactive: dict_clustering_2d_multimodel.html

MULTI-MODEL COMPARISON COMPLETE


In [26]:
# ============================================================
# CELL 9.10: DICTIONARY TERM CLUSTERING - 3D Exploration
# ============================================================

if VIZ_AVAILABLE and len(dict_embeddings) > 0:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.10: Dictionary Term Clustering - 3D Exploration")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Choose Model for 3D Visualization
    # --------------------------------------------------------

    print(f"\n1️⃣  Selecting model for 3D exploration...")

    # Prefer policy_trained, fallback to first available
    if 'policy_trained' in dict_embeddings:
        model_3d = 'policy_trained'
    else:
        model_3d = list(dict_embeddings.keys())[0]

    print(f"   Using model: {model_3d}")

    embeddings_3d = dict_embeddings[model_3d]

    # --------------------------------------------------------
    # 2. Perform 3D PCA
    # --------------------------------------------------------

    print(f"\n2️⃣  Performing 3D PCA...")

    pca_3d = PCA(n_components=3, random_state=PCA_RANDOM_STATE)
    coords_3d = pca_3d.fit_transform(embeddings_3d)

    var_explained = pca_3d.explained_variance_ratio_
    total_var = var_explained.sum()

    print(f"   PC1: {var_explained[0]:.1%}")
    print(f"   PC2: {var_explained[1]:.1%}")
    print(f"   PC3: {var_explained[2]:.1%}")
    print(f"   Total: {total_var:.1%}")

    # --------------------------------------------------------
    # 3. Prepare DataFrame for Plotting
    # --------------------------------------------------------

    df_dict_3d = df_dict.copy()
    df_dict_3d['x'] = coords_3d[:, 0]
    df_dict_3d['y'] = coords_3d[:, 1]
    df_dict_3d['z'] = coords_3d[:, 2]

    # --------------------------------------------------------
    # 4. Create 3D Scatter Plot
    # --------------------------------------------------------

    print(f"\n3️⃣  Creating 3D visualization...")

    # Color palette
    colors = px.colors.qualitative.Set2
    if len(topics) > len(colors):
        colors = px.colors.qualitative.Alphabet
    topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

    fig = go.Figure()

    # Add scatter for each topic
    for topic in topics:
        topic_data = df_dict_3d[df_dict_3d['topic'] == topic]

        # Prepare hover text with term details
        hover_texts = []
        for _, row in topic_data.iterrows():
            text_parts = [f"<b>{row['term']}</b>"]
            text_parts.append(f"Topic: {topic}")
            if 'weight' in row and pd.notna(row['weight']):
                text_parts.append(f"Weight: {row['weight']:.2f}")
            if 'category' in row and pd.notna(row['category']):
                text_parts.append(f"Category: {row['category']}")
            if 'is_seed' in row:
                text_parts.append(f"Type: {'Seed' if row['is_seed'] else 'Expanded'}")

            hover_texts.append('<br>'.join(text_parts))

        fig.add_trace(go.Scatter3d(
            x=topic_data['x'],
            y=topic_data['y'],
            z=topic_data['z'],
            mode='markers+text',
            name=topic,
            marker=dict(
                size=6,
                color=topic_colors[topic],
                opacity=0.8,
                line=dict(width=0.5, color='white')
            ),
            text=topic_data['term'],
            textposition='top center',
            textfont=dict(size=8),
            hovertext=hover_texts,
            hoverinfo='text',
            showlegend=True
        ))

    # Update layout
    fig.update_layout(
        title={
            'text': f"Dictionary Term Clustering - 3D Interactive Exploration<br><sub>Model: {model_3d} | Total variance: {total_var:.1%}</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        scene=dict(
            xaxis_title=f"PC1 ({var_explained[0]:.1%})",
            yaxis_title=f"PC2 ({var_explained[1]:.1%})",
            zaxis_title=f"PC3 ({var_explained[2]:.1%})",
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.3)
            )
        ),
        height=800,
        template='plotly_white',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=0,
            title="Topics"
        )
    )

    # --------------------------------------------------------
    # 5. Display & Save
    # --------------------------------------------------------

    if SHOW_IN_NOTEBOOK:
        section3_figs.append(fig)  # Store for export
        fig.show()

    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'dict_clustering_3d.html'
        fig.write_html(str(output_path))
        print(f"   ✓ Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'dict_clustering_3d.png'
        fig.write_image(str(output_path), width=1400, height=1000, scale=2)
        print(f"   ✓ Saved static: {output_path.name}")

    print(f"\n{'='*70}")
    print("3D EXPLORATION COMPLETE")
    print(f"{'='*70}")
    print(f"\n✅ Section 3 Complete - Dictionary Fitness Validated!")

else:
    print("\n⊘ Skipping 3D exploration - no embeddings available")


VISUALIZATION 9.10: Dictionary Term Clustering - 3D Exploration

1️⃣  Selecting model for 3D exploration...
   Using model: pretrained_bertje

2️⃣  Performing 3D PCA...
   PC1: 35.2%
   PC2: 22.2%
   PC3: 4.7%
   Total: 62.1%

3️⃣  Creating 3D visualization...


   ✓ Saved interactive: dict_clustering_3d.html

3D EXPLORATION COMPLETE

✅ Section 3 Complete - Dictionary Fitness Validated!


In [27]:
# ============================================================
# SECTION 3: EXPORT VISUALIZATIONS
# ============================================================
from pathlib import Path

# Create exports directory if it doesn't exist
export_dir = Path(SOURCE_WORKFLOW) / 'Visualizations'
export_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "="*70)
print(f"EXPORTING SECTION 3 VISUALIZATIONS")
print("="*70)

# Section 3 visualizations:
# 1. Weight Tier Validation (Box plot)
# 2. Expansion Quality (2D PCA with seeds vs expanded)
# 3. Dictionary Term Clustering - Model Comparison (2D PCA subplots)
# 4. Dictionary Term Clustering - 3D Interactive

try:
    # Check if figures were stored in section3_figs list
    if 'section3_figs' in globals() and len(section3_figs) > 0:
        # Export each figure
        fig_names = [
        'section3_weight_tier_validation',
        'section3_expansion_quality_2d',
        'section3_clustering_model_comparison',
        'section3_clustering_3d_interactive',
        ]

        for i, (fig, name) in enumerate(zip(section3_figs, fig_names)):
            # Save as HTML (interactive)
            html_path = export_dir / f'{name}.html'
            fig.write_html(str(html_path))
            print(f"✓ Saved: {html_path}")

            # Save as PNG (static image) - requires kaleido package
            try:
                png_path = export_dir / f'{name}.png'
                # Adjust size based on visualization type
                if '3d' in name or 'matrix' in name:
                    fig.write_image(str(png_path), width=1000, height=1000, scale=2)
                elif 'comparison' in name or 'heatmaps' in name:
                    fig.write_image(str(png_path), width=1400, height=800, scale=2)
                else:
                    fig.write_image(str(png_path), width=1200, height=800, scale=2)
                print(f"✓ Saved: {png_path}")
            except Exception as e:
                print(f"⚠️  PNG export failed: {e}")
                print(f"   (Install kaleido: pip install -U kaleido)")

        print(f"\n✓ Exported {len(section3_figs)} visualizations from Section 3")
    else:
        print(f"\nℹ️  No figures found in 'section3_figs' variable.")
        print("    To enable export, figures need to be stored in the list.")

except Exception as e:
    print(f"⚠️  Export error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)
print(f"Section 3 export complete.")
print("="*70)



EXPORTING SECTION 3 VISUALIZATIONS
✓ Saved: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Visualizations\section3_weight_tier_validation.html
✓ Saved: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Visualizations\section3_weight_tier_validation.png
✓ Saved: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Visualizations\section3_expansion_quality_2d.html
✓ Saved: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Visualizations\section3_expansion_quality_2d.png
✓ Saved: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Visualizations\section3_clustering_model_comparison.html
✓ Saved: C:\Users\Home\policy-analysis\workflow_reneweddict\slavery_structured-slavdict_pretrained_slavery_v1\Visualizations\section3_clustering_model_comp

---
## SECTION 4: Topic Coherence & Model Performance
---

**Goal**: Quantitatively validate that model training improves topic separation.

**Key Questions**:
1. Do topics form distinct clusters (silhouette score, CH index)?
2. Which topics improve most with training?
3. Which topic pairs are most confused?
4. Did training converge successfully?

**Visualizations**:
- **9.11**: Cluster Quality Metrics Table & Chart
- **9.12**: Topic Separation Heatmap (Confusion Matrix)
- **9.13**: Training Metrics Timeline (if available)

In [29]:
# ============================================================
# Initialize figure storage for Section 4
# ============================================================
section4_figs = []
print(f'Initialized section4_figs list for storing visualizations')


Initialized section4_figs list for storing visualizations


In [30]:
# ============================================================
# CELL 9.11: CLUSTER QUALITY METRICS TABLE & CHART
# ============================================================

if VIZ_AVAILABLE and len(dict_embeddings) > 0:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.11: Cluster Quality Metrics")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Calculate Metrics for Each Model
    # --------------------------------------------------------

    print(f"\n1️⃣  Calculating cluster quality metrics...")

    metrics_results = []

    # Get topic labels for all dictionary terms
    topic_labels = df_dict['topic'].values

    for model_name in ['pretrained_bertje', 'slavery_trained', 'policy_trained']:
        if model_name not in dict_embeddings:
            continue

        print(f"\n{model_name}:")

        embeddings = dict_embeddings[model_name]

        # === Silhouette Score ===
        # Measures how similar each point is to its own cluster vs. other clusters
        # Range: [-1, 1], higher = better separation
        sil_score = silhouette_score(embeddings, topic_labels)
        print(f"  Silhouette Score: {sil_score:.3f}")

        # === Calinski-Harabasz Index ===
        # Ratio of between-cluster to within-cluster variance
        # Higher = more distinct clusters
        ch_score = calinski_harabasz_score(embeddings, topic_labels)
        print(f"  Calinski-Harabasz: {ch_score:.1f}")

        # === Average Intra-Topic Distance ===
        # Mean distance from each term to its topic centroid
        intra_distances = []

        for topic in topics:
            topic_mask = df_dict['topic'] == topic
            topic_embeddings = embeddings[topic_mask]

            if len(topic_embeddings) < 2:
                continue

            # Calculate centroid
            centroid = topic_embeddings.mean(axis=0)

            # Calculate distances to centroid
            distances = np.linalg.norm(topic_embeddings - centroid, axis=1)
            intra_distances.extend(distances.tolist())

        avg_intra_distance = np.mean(intra_distances)
        print(f"  Avg Intra-Topic Distance: {avg_intra_distance:.3f}")

        # === Per-Topic Tightness ===
        topic_tightness = {}

        for topic in topics:
            topic_mask = df_dict['topic'] == topic
            topic_embeddings = embeddings[topic_mask]

            if len(topic_embeddings) < 2:
                topic_tightness[topic] = np.nan
                continue

            centroid = topic_embeddings.mean(axis=0)
            distances = np.linalg.norm(topic_embeddings - centroid, axis=1)
            topic_tightness[topic] = distances.mean()

        # Store results
        metrics_results.append({
            'model': model_name,
            'silhouette': sil_score,
            'calinski_harabasz': ch_score,
            'avg_intra_distance': avg_intra_distance,
            'topic_tightness': topic_tightness
        })

    if len(metrics_results) == 0:
        print("\n   ⚠ No models available for metrics calculation")
        VIZ_AVAILABLE = False
    else:
        print(f"\n   ✓ Calculated metrics for {len(metrics_results)} models")

        # --------------------------------------------------------
        # 2. Create Overall Metrics Table
        # --------------------------------------------------------

        print(f"\n2️⃣  Creating metrics comparison table...")

        df_metrics = pd.DataFrame(metrics_results)

        # Calculate improvement (if we have baseline)
        if 'pretrained_bertje' in df_metrics['model'].values:
            baseline_idx = df_metrics[df_metrics['model'] == 'pretrained_bertje'].index[0]
            baseline_sil = df_metrics.loc[baseline_idx, 'silhouette']
            baseline_ch = df_metrics.loc[baseline_idx, 'calinski_harabasz']
            baseline_dist = df_metrics.loc[baseline_idx, 'avg_intra_distance']

            df_metrics['silhouette_improvement'] = ((df_metrics['silhouette'] / baseline_sil) - 1) * 100
            df_metrics['ch_improvement'] = ((df_metrics['calinski_harabasz'] / baseline_ch) - 1) * 100
            df_metrics['distance_reduction'] = ((df_metrics['avg_intra_distance'] / baseline_dist) - 1) * 100
        else:
            df_metrics['silhouette_improvement'] = 0
            df_metrics['ch_improvement'] = 0
            df_metrics['distance_reduction'] = 0

        # --------------------------------------------------------
        # 3. Create Visualization - Metrics Bar Chart
        # --------------------------------------------------------

        print(f"\n3️⃣  Creating metrics visualization...")

        # Prepare data for plotting (normalize for comparison)
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=("Silhouette Score<br>(higher = better)",
                          "Calinski-Harabasz Index<br>(higher = better)",
                          "Avg Intra-Topic Distance<br>(lower = better)"),
            horizontal_spacing=0.12
        )

        colors_map = {
            'pretrained_bertje': '#1f77b4',
            'slavery_trained': '#ff7f0e',
            'policy_trained': '#2ca02c'
        }

        model_labels = {
            'pretrained_bertje': 'Pretrained',
            'slavery_trained': 'Slavery-trained',
            'policy_trained': 'Policy-trained'
        }

        # Silhouette Score
        fig.add_trace(
            go.Bar(
                x=[model_labels.get(m, m) for m in df_metrics['model']],
                y=df_metrics['silhouette'],
                name='Silhouette',
                marker_color=[colors_map.get(m, 'gray') for m in df_metrics['model']],
                text=[f"{v:.3f}" for v in df_metrics['silhouette']],
                textposition='outside',
                showlegend=False
            ),
            row=1, col=1
        )

        # Calinski-Harabasz
        fig.add_trace(
            go.Bar(
                x=[model_labels.get(m, m) for m in df_metrics['model']],
                y=df_metrics['calinski_harabasz'],
                name='CH Index',
                marker_color=[colors_map.get(m, 'gray') for m in df_metrics['model']],
                text=[f"{v:.0f}" for v in df_metrics['calinski_harabasz']],
                textposition='outside',
                showlegend=False
            ),
            row=1, col=2
        )

        # Intra-Topic Distance
        fig.add_trace(
            go.Bar(
                x=[model_labels.get(m, m) for m in df_metrics['model']],
                y=df_metrics['avg_intra_distance'],
                name='Intra-Distance',
                marker_color=[colors_map.get(m, 'gray') for m in df_metrics['model']],
                text=[f"{v:.3f}" for v in df_metrics['avg_intra_distance']],
                textposition='outside',
                showlegend=False
            ),
            row=1, col=3
        )

        # Update layout
        fig.update_layout(
            title={
                'text': "Cluster Quality Metrics: Model Comparison<br><sub>Higher silhouette/CH and lower distance indicate better topic separation</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            height=500,
            template='plotly_white'
        )

        # Update y-axes
        fig.update_yaxes(title_text="Score", row=1, col=1)
        fig.update_yaxes(title_text="Index", row=1, col=2)
        fig.update_yaxes(title_text="Distance", row=1, col=3)

        # --------------------------------------------------------
        # 4. Display & Save
        # --------------------------------------------------------

        if SHOW_IN_NOTEBOOK:
            section4_figs.append(fig)  # Store for export
            fig.show()

        if SAVE_INTERACTIVE:
            output_path = visuals_dir / 'cluster_quality_metrics.html'
            fig.write_html(str(output_path))
            print(f"   ✓ Saved interactive: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / 'cluster_quality_metrics.png'
            fig.write_image(str(output_path), width=1600, height=600, scale=2)
            print(f"   ✓ Saved static: {output_path.name}")

        # Save metrics table
        metrics_table_path = visuals_dir / 'cluster_metrics_table.csv'
        df_metrics.to_csv(metrics_table_path, index=False)
        print(f"   ✓ Saved table: {metrics_table_path.name}")

        # --------------------------------------------------------
        # 5. Per-Topic Tightness Comparison
        # --------------------------------------------------------

        print(f"\n4️⃣  Per-topic tightness comparison...")

        # Build per-topic dataframe
        topic_tightness_data = []

        for result in metrics_results:
            model_name = result['model']
            for topic, tightness in result['topic_tightness'].items():
                topic_tightness_data.append({
                    'model': model_name,
                    'topic': topic,
                    'tightness': tightness
                })

        df_topic_tightness = pd.DataFrame(topic_tightness_data)

        # Pivot for easier comparison
        df_pivot = df_topic_tightness.pivot(index='topic', columns='model', values='tightness')

        # Calculate improvement (if baseline exists)
        if 'pretrained_bertje' in df_pivot.columns:
            for col in df_pivot.columns:
                if col != 'pretrained_bertje':
                    improvement_col = f'{col}_improvement'
                    df_pivot[improvement_col] = ((df_pivot[col] / df_pivot['pretrained_bertje']) - 1) * 100

        # Save per-topic table
        topic_table_path = visuals_dir / 'per_topic_tightness.csv'
        df_pivot.to_csv(topic_table_path)
        print(f"   ✓ Saved per-topic table: {topic_table_path.name}")

        # --------------------------------------------------------
        # 6. Print Summary
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("CLUSTER QUALITY SUMMARY")
        print(f"{'='*70}")

        for _, row in df_metrics.iterrows():
            print(f"\n{model_labels.get(row['model'], row['model'])}:")
            print(f"  Silhouette Score:        {row['silhouette']:.3f}", end="")
            if row['silhouette_improvement'] != 0:
                print(f"  ({row['silhouette_improvement']:+.1f}% vs. pretrained)")
            else:
                print()

            print(f"  Calinski-Harabasz:       {row['calinski_harabasz']:.0f}", end="")
            if row['ch_improvement'] != 0:
                print(f"  ({row['ch_improvement']:+.1f}% vs. pretrained)")
            else:
                print()

            print(f"  Avg Intra-Topic Dist:    {row['avg_intra_distance']:.3f}", end="")
            if row['distance_reduction'] != 0:
                print(f"  ({row['distance_reduction']:+.1f}% vs. pretrained)")
            else:
                print()

        print(f"\n{'='*70}")

else:
    print("\n⊘ Skipping cluster quality metrics - no embeddings available")


VISUALIZATION 9.11: Cluster Quality Metrics

1️⃣  Calculating cluster quality metrics...

pretrained_bertje:
  Silhouette Score: 0.422
  Calinski-Harabasz: 497.9
  Avg Intra-Topic Distance: 3.483

slavery_trained:
  Silhouette Score: 0.317
  Calinski-Harabasz: 286.6
  Avg Intra-Topic Distance: 8.189

   ✓ Calculated metrics for 2 models

2️⃣  Creating metrics comparison table...

3️⃣  Creating metrics visualization...


   ✓ Saved interactive: cluster_quality_metrics.html
   ✓ Saved table: cluster_metrics_table.csv

4️⃣  Per-topic tightness comparison...
   ✓ Saved per-topic table: per_topic_tightness.csv

CLUSTER QUALITY SUMMARY

Pretrained:
  Silhouette Score:        0.422
  Calinski-Harabasz:       498
  Avg Intra-Topic Dist:    3.483

Slavery-trained:
  Silhouette Score:        0.317  (-24.9% vs. pretrained)
  Calinski-Harabasz:       287  (-42.4% vs. pretrained)
  Avg Intra-Topic Dist:    8.189  (+135.1% vs. pretrained)



In [31]:
# ============================================================
# CELL 9.12: TOPIC SEPARATION HEATMAP (Confusion Matrix)
# ============================================================

if VIZ_AVAILABLE and len(dict_embeddings) > 0:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.12: Topic Separation Heatmap")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Calculate Inter-Topic Similarity Matrices
    # --------------------------------------------------------

    print(f"\n1️⃣  Calculating inter-topic similarities...")

    similarity_matrices = {}

    for model_name in ['pretrained_bertje', 'slavery_trained', 'policy_trained']:
        if model_name not in dict_embeddings:
            continue

        print(f"\n{model_name}:")

        embeddings = dict_embeddings[model_name]

        # Calculate topic centroids
        topic_centroids = {}
        for topic in topics:
            topic_mask = df_dict['topic'] == topic
            topic_embeddings = embeddings[topic_mask]

            if len(topic_embeddings) > 0:
                topic_centroids[topic] = topic_embeddings.mean(axis=0)

        # Calculate pairwise cosine similarities between centroids
        n_topics = len(topics)
        similarity_matrix = np.zeros((n_topics, n_topics))

        for i, topic_i in enumerate(topics):
            for j, topic_j in enumerate(topics):
                if topic_i in topic_centroids and topic_j in topic_centroids:
                    centroid_i = topic_centroids[topic_i].reshape(1, -1)
                    centroid_j = topic_centroids[topic_j].reshape(1, -1)

                    sim = cosine_similarity(centroid_i, centroid_j)[0, 0]
                    similarity_matrix[i, j] = sim

        similarity_matrices[model_name] = similarity_matrix

        # Report most confused topic pairs
        # (excluding diagonal, find highest similarities)
        off_diagonal_mask = ~np.eye(n_topics, dtype=bool)
        off_diagonal_sims = similarity_matrix[off_diagonal_mask]

        top_confusions_idx = np.argsort(off_diagonal_sims)[-3:][::-1]

        # Map back to topic pairs
        confusion_pairs = []
        idx = 0
        for i in range(n_topics):
            for j in range(n_topics):
                if i != j:
                    if idx in top_confusions_idx:
                        confusion_pairs.append((topics[i], topics[j], similarity_matrix[i, j]))
                    idx += 1

        print(f"  Top 3 most confused topic pairs:")
        for topic_i, topic_j, sim in confusion_pairs:
            print(f"    {topic_i[:30]:30s} ↔ {topic_j[:30]:30s}: {sim:.3f}")

    if len(similarity_matrices) == 0:
        print("\n   ⚠ No models available for separation analysis")
        VIZ_AVAILABLE = False
    else:
        print(f"\n   ✓ Calculated similarity matrices for {len(similarity_matrices)} models")

        # --------------------------------------------------------
        # 2. Create Heatmap Subplots
        # --------------------------------------------------------

        print(f"\n2️⃣  Creating heatmap visualization...")

        n_models = len(similarity_matrices)

        fig = make_subplots(
            rows=1, cols=n_models,
            subplot_titles=[m.replace('_', ' ').title() for m in similarity_matrices.keys()],
            horizontal_spacing=0.12
        )

        model_labels = {
            'pretrained_bertje': 'Pretrained',
            'slavery_trained': 'Slavery-trained',
            'policy_trained': 'Policy-trained'
        }

        # Shorten topic labels for display
        topic_labels_short = [t[:25] + '...' if len(t) > 25 else t for t in topics]

        for col_idx, (model_name, sim_matrix) in enumerate(similarity_matrices.items(), start=1):
            # Create heatmap
            heatmap = go.Heatmap(
                z=sim_matrix,
                x=topic_labels_short,
                y=topic_labels_short,
                colorscale='RdYlGn_r',  # Red = high similarity (confused), Green = low (distinct)
                zmin=0,
                zmax=1,
                text=np.round(sim_matrix, 2),
                texttemplate='%{text}',
                textfont={"size": 8},
                colorbar=dict(
                    title="Similarity",
                    x=1.02 if col_idx == n_models else None,
                    len=0.9
                ) if col_idx == n_models else None,
                showscale=(col_idx == n_models),  # Only show colorbar on last subplot
                hovertemplate='%{y} ↔ %{x}<br>Similarity: %{z:.3f}<extra></extra>'
            )

            fig.add_trace(heatmap, row=1, col=col_idx)

            # Update axes
            fig.update_xaxes(tickangle=45, row=1, col=col_idx)

        # Update layout
        fig.update_layout(
            title={
                'text': "Topic Separation: Inter-Topic Centroid Similarity<br><sub>Lower similarity (green) = better separation | Diagonal = 1.0 (self-similarity)</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            height=600,
            template='plotly_white'
        )

        # --------------------------------------------------------
        # 3. Display & Save
        # --------------------------------------------------------

        if SHOW_IN_NOTEBOOK:
            section4_figs.append(fig)  # Store for export
            fig.show()

        if SAVE_INTERACTIVE:
            output_path = visuals_dir / 'topic_separation_heatmap.html'
            fig.write_html(str(output_path))
            print(f"   ✓ Saved interactive: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / 'topic_separation_heatmap.png'
            fig.write_image(str(output_path), width=1800, height=800, scale=2)
            print(f"   ✓ Saved static: {output_path.name}")

        # --------------------------------------------------------
        # 4. Calculate Separation Improvement
        # --------------------------------------------------------

        if 'pretrained_bertje' in similarity_matrices and 'policy_trained' in similarity_matrices:
            print(f"\n3️⃣  Analyzing separation improvement...")

            pre_matrix = similarity_matrices['pretrained_bertje']
            post_matrix = similarity_matrices['policy_trained']

            # Calculate average off-diagonal similarity (lower = better separation)
            n = len(topics)
            off_diag_mask = ~np.eye(n, dtype=bool)

            pre_avg_sim = pre_matrix[off_diag_mask].mean()
            post_avg_sim = post_matrix[off_diag_mask].mean()

            separation_improvement = ((post_avg_sim / pre_avg_sim) - 1) * 100

            print(f"\n   Pretrained avg inter-topic similarity: {pre_avg_sim:.3f}")
            print(f"   Policy-trained avg inter-topic similarity: {post_avg_sim:.3f}")
            print(f"   Separation improvement: {separation_improvement:+.1f}%")

            if separation_improvement < -10:
                print(f"   ✓ GOOD: Topics more distinct after training")
            elif separation_improvement < 0:
                print(f"   ⚠ MODERATE: Slight improvement")
            else:
                print(f"   ❌ CONCERN: Topics less distinct after training")

        print(f"\n{'='*70}")
        print("TOPIC SEPARATION COMPLETE")
        print(f"{'='*70}")

else:
    print("\n⊘ Skipping topic separation heatmap - no embeddings available")


VISUALIZATION 9.12: Topic Separation Heatmap

1️⃣  Calculating inter-topic similarities...

pretrained_bertje:
  Top 3 most confused topic pairs:
    Contemporary_Manifestations    ↔ Structural_Continuity_Neocolon: 0.895
    Historical_Slavery_Colonialism ↔ Contemporary_Manifestations   : 0.889
    Structural_Continuity_Neocolon ↔ Contemporary_Manifestations   : 0.895

slavery_trained:
  Top 3 most confused topic pairs:
    Contemporary_Manifestations    ↔ Historical_Slavery_Colonialism: 0.647
    Historical_Slavery_Colonialism ↔ Contemporary_Manifestations   : 0.647
    Structural_Continuity_Neocolon ↔ Historical_Slavery_Colonialism: 0.629

   ✓ Calculated similarity matrices for 2 models

2️⃣  Creating heatmap visualization...


   ✓ Saved interactive: topic_separation_heatmap.html

TOPIC SEPARATION COMPLETE


In [49]:
# ============================================================
# CELL 9.13: TRAINING METRICS VISUALIZATION (if available)
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.13: Training Metrics Timeline")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Check for Training Metrics File
    # --------------------------------------------------------

    print(f"\n1️⃣  Looking for training metrics...")

    training_metrics_path = folders.get('Model_finetuning') / 'training_metrics.json'

    if not training_metrics_path.exists():
        print(f"   ⚠ Training metrics not found at: {training_metrics_path}")
        print(f"   ⊘ Skipping training metrics visualization")
        print(f"\n   Note: This is expected if model training was done outside this workflow")
    else:
        print(f"   ✓ Found training metrics: {training_metrics_path.name}")

        # --------------------------------------------------------
        # 2. Load Training Metrics
        # --------------------------------------------------------

        print(f"\n2️⃣  Loading training data...")

        with open(training_metrics_path, 'r', encoding='utf-8') as f:
            training_data = json.load(f)

        # Expected structure (from CP7):
        # {
        #   'epoch': [1, 2, 3, ...],
        #   'train_loss': [...],
        #   'eval_loss': [...],
        #   'eval_accuracy': [...],  # if available
        #   'learning_rate': [...]   # if available
        # }

        available_metrics = list(training_data.keys())
        print(f"   Available metrics: {', '.join(available_metrics)}")

        # --------------------------------------------------------
        # 3. Create Training Timeline Visualization
        # --------------------------------------------------------

        print(f"\n3️⃣  Creating training timeline...")

        # Determine subplot layout based on available metrics
        has_eval = 'eval_loss' in training_data or 'eval_accuracy' in training_data
        n_subplots = 2 if has_eval else 1

        if n_subplots == 2:
            fig = make_subplots(
                rows=1, cols=2,
                subplot_titles=("Training & Validation Loss", "Learning Rate (if available)"),
                horizontal_spacing=0.12
            )
        else:
            fig = go.Figure()

        epochs = training_data.get('epoch', list(range(1, len(training_data.get('train_loss', [])) + 1)))

        # === Training Loss ===
        if 'train_loss' in training_data:
            fig.add_trace(
                go.Scatter(
                    x=epochs,
                    y=training_data['train_loss'],
                    mode='lines+markers',
                    name='Training Loss',
                    line=dict(color='#1f77b4', width=2),
                    marker=dict(size=6)
                ),
                row=1, col=1 if n_subplots == 2 else None
            )

        # === Validation Loss ===
        if 'eval_loss' in training_data:
            fig.add_trace(
                go.Scatter(
                    x=epochs,
                    y=training_data['eval_loss'],
                    mode='lines+markers',
                    name='Validation Loss',
                    line=dict(color='#ff7f0e', width=2, dash='dash'),
                    marker=dict(size=6)
                ),
                row=1, col=1 if n_subplots == 2 else None
            )

        # === Learning Rate ===
        if 'learning_rate' in training_data and n_subplots == 2:
            fig.add_trace(
                go.Scatter(
                    x=epochs,
                    y=training_data['learning_rate'],
                    mode='lines',
                    name='Learning Rate',
                    line=dict(color='#2ca02c', width=2),
                    yaxis='y2'
                ),
                row=1, col=2
            )

        # Update layout
        if n_subplots == 2:
            fig.update_xaxes(title_text="Epoch", row=1, col=1)
            fig.update_xaxes(title_text="Epoch", row=1, col=2)
            fig.update_yaxes(title_text="Loss", row=1, col=1)
            fig.update_yaxes(title_text="Learning Rate", row=1, col=2)
        else:
            fig.update_xaxes(title_text="Epoch")
            fig.update_yaxes(title_text="Loss")

        fig.update_layout(
            title={
                'text': "Model Training Metrics: Loss & Learning Rate Over Time<br><sub>Policy-trained model (V10 finetuning)</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            height=500,
            hovermode='x unified',
            template='plotly_white',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )

        # --------------------------------------------------------
        # 4. Display & Save
        # --------------------------------------------------------

        if SHOW_IN_NOTEBOOK:
            section4_figs.append(fig)  # Store for export
            fig.show()

        if SAVE_INTERACTIVE:
            output_path = visuals_dir / 'training_metrics.html'
            fig.write_html(str(output_path))
            print(f"   ✓ Saved interactive: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / 'training_metrics.png'
            fig.write_image(str(output_path), width=1400, height=600, scale=2)
            print(f"   ✓ Saved static: {output_path.name}")

        # --------------------------------------------------------
        # 5. Training Summary
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("TRAINING METRICS SUMMARY")
        print(f"{'='*70}")

        n_epochs = len(epochs)
        print(f"\nTotal epochs: {n_epochs}")

        if 'train_loss' in training_data:
            initial_loss = training_data['train_loss'][0]
            final_loss = training_data['train_loss'][-1]
            loss_reduction = ((final_loss / initial_loss) - 1) * 100

            print(f"\nTraining Loss:")
            print(f"  Initial (epoch 1): {initial_loss:.4f}")
            print(f"  Final (epoch {n_epochs}): {final_loss:.4f}")
            print(f"  Reduction: {loss_reduction:+.1f}%")

        if 'eval_loss' in training_data:
            initial_eval = training_data['eval_loss'][0]
            final_eval = training_data['eval_loss'][-1]
            eval_reduction = ((final_eval / initial_eval) - 1) * 100

            print(f"\nValidation Loss:")
            print(f"  Initial: {initial_eval:.4f}")
            print(f"  Final: {final_eval:.4f}")
            print(f"  Reduction: {eval_reduction:+.1f}%")

            # Check for overfitting
            train_val_gap = final_loss - final_eval if 'train_loss' in training_data else 0
            if train_val_gap > 0.1:
                print(f"\n  ⚠ WARNING: Train-validation gap = {train_val_gap:.4f} (possible overfitting)")
            else:
                print(f"\n  ✓ Train-validation gap within acceptable range")

        print(f"\n{'='*70}")
        print(f"\n✅ Section 4 Complete - Topic Coherence Validated!")

else:
    print("\n⊘ Skipping training metrics - VIZ_AVAILABLE = False")


VISUALIZATION 9.13: Training Metrics Timeline

1️⃣  Looking for training metrics...
   ✓ Found training metrics: training_metrics.json

2️⃣  Loading training data...
   Available metrics: final_eval, training_history, topics, num_train_samples, num_val_samples, training_args

3️⃣  Creating training timeline...


   ✓ Saved interactive: training_metrics.html

TRAINING METRICS SUMMARY

Total epochs: 0


✅ Section 4 Complete - Topic Coherence Validated!


In [48]:
# ============================================================
# SECTION 4: EXPORT VISUALIZATIONS
# ============================================================
from pathlib import Path

# Create exports directory if it doesn't exist
export_dir = Path(SOURCE_WORKFLOW) / 'Visualizations'
export_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "="*70)
print(f"EXPORTING SECTION 4 VISUALIZATIONS")
print("="*70)

# Section 4 visualizations:
# 1. Cluster Quality Metrics - Model Comparison
# 2. Topic Separation - Inter-Topic Centroid Similarity heatmaps
# 3. Model Training Metrics (Loss & Learning Rate)

try:
    # Check if figures were stored in section4_figs list
    if 'section4_figs' in globals() and len(section4_figs) > 0:
        # Export each figure
        fig_names = [
        'section4_cluster_quality_metrics',
        'section4_topic_separation_heatmaps',
        'section4_training_metrics',
        ]

        for i, (fig, name) in enumerate(zip(section4_figs, fig_names)):
            # Save as HTML (interactive)
            html_path = export_dir / f'{name}.html'
            fig.write_html(str(html_path))
            print(f"✓ Saved: {html_path}")

            # Save as PNG (static image) - requires kaleido package
            try:
                png_path = export_dir / f'{name}.png'
                # Adjust size based on visualization type
                if '3d' in name or 'matrix' in name:
                    fig.write_image(str(png_path), width=1000, height=1000, scale=2)
                elif 'comparison' in name or 'heatmaps' in name:
                    fig.write_image(str(png_path), width=1400, height=800, scale=2)
                else:
                    fig.write_image(str(png_path), width=1200, height=800, scale=2)
                print(f"✓ Saved: {png_path}")
            except Exception as e:
                print(f"⚠️  PNG export failed: {e}")
                print(f"   (Install kaleido: pip install -U kaleido)")

        print(f"\n✓ Exported {len(section4_figs)} visualizations from Section 4")
    else:
        print(f"\nℹ️  No figures found in 'section4_figs' variable.")
        print("    To enable export, figures need to be stored in the list.")

except Exception as e:
    print(f"⚠️  Export error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)
print(f"Section 4 export complete.")
print("="*70)



EXPORTING SECTION 4 VISUALIZATIONS

ℹ️  No figures found in 'section4_figs' variable.
    To enable export, figures need to be stored in the list.

Section 4 export complete.


---
## SECTION 5: Chunk Scoring Analysis
---

**Goal**: Analyze how chunks cluster and score across different models.

**Key Questions**:
1. How do chunks cluster in semantic space?
2. Does finetuning shift chunk positions toward better topic alignment?
3. What are the score distributions by topic and model?
4. Which chunks show largest shifts (and why)?

**Visualizations**:
- **9.14**: PCA Preparation for Chunk Analysis
- **9.15**: Chunk Clustering 2D - Multi-Model Comparison
- **9.16**: Chunk Clustering 3D - Interactive Exploration
- **9.17**: Chunk Shift Analysis - Pre/Post Training
- **9.18**: Chunk Shift Vectors (Top Shifters)
- **9.19**: Score Distribution by Topic (Violin Plots)
- **9.20**: Multi-Label Distribution Analysis

In [32]:
# ============================================================
# Initialize figure storage for Section 5
# ============================================================
section5_figs = []
print(f'Initialized section5_figs list for storing visualizations')


Initialized section5_figs list for storing visualizations


In [ ]:
# ============================================================
# CELL 9.14: PCA PREPARATION FOR CHUNK ANALYSIS
# ============================================================

if VIZ_AVAILABLE and len(chunk_embeddings) > 0:
    print(f"\n{'='*70}")
    print("CELL 9.14: PCA Preparation for Chunk Analysis")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Prepare Chunk Metadata
    # --------------------------------------------------------

    print(f"\n1️⃣  Preparing chunk metadata...")

    # Ensure df_chunks_sampled has primary topic assignments
    if 'primary_topic' not in df_chunks_sampled.columns:
        # Assign based on highest score
        primary_topic_idx = df_chunks_sampled[score_cols].values.argmax(axis=1)
        df_chunks_sampled['primary_topic'] = [topics[i] for i in primary_topic_idx]

    # Get max score for each chunk (confidence level)
    df_chunks_sampled['max_score'] = df_chunks_sampled[score_cols].max(axis=1)

    # Count how many topics exceed threshold (multi-label indicator)
    threshold = MIN_SCORE_THRESHOLD
    df_chunks_sampled['n_topics_above_threshold'] = (df_chunks_sampled[score_cols] >= threshold).sum(axis=1)

    print(f"   ✓ Chunks: {len(df_chunks_sampled)}")
    print(f"   ✓ Primary topic distribution:")
    for topic in topics:
        count = (df_chunks_sampled['primary_topic'] == topic).sum()
        pct = count / len(df_chunks_sampled) * 100
        print(f"     {topic[:45]:45s}: {count:4d} ({pct:5.1f}%)")

    # --------------------------------------------------------
    # 2. Perform PCA for Each Model (2D and 3D)
    # --------------------------------------------------------

    print(f"\n2️⃣  Performing PCA (2D and 3D)...")

    chunk_pca_2d = {}
    chunk_pca_3d = {}

    for model_name in chunk_embeddings.keys():
        print(f"\n{model_name}:")

        embeddings = chunk_embeddings[model_name]

        # 2D PCA
        pca_2d = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
        coords_2d = pca_2d.fit_transform(embeddings)
        chunk_pca_2d[model_name] = {
            'coords': coords_2d,
            'pca': pca_2d,
            'var_explained': pca_2d.explained_variance_ratio_
        }

        print(f"  2D PCA: {pca_2d.explained_variance_ratio_.sum():.1%} variance explained")

        # 3D PCA
        pca_3d = PCA(n_components=3, random_state=PCA_RANDOM_STATE)
        coords_3d = pca_3d.fit_transform(embeddings)
        chunk_pca_3d[model_name] = {
            'coords': coords_3d,
            'pca': pca_3d,
            'var_explained': pca_3d.explained_variance_ratio_
        }

        print(f"  3D PCA: {pca_3d.explained_variance_ratio_.sum():.1%} variance explained")

    print(f"\n   ✓ PCA complete for {len(chunk_pca_2d)} models")

    # --------------------------------------------------------
    # 3. Add PCA Coordinates to DataFrame (for policy_trained)
    # --------------------------------------------------------

    print(f"\n3️⃣  Adding coordinates to dataframe...")

    # Use policy_trained if available, else first model
    primary_model = 'policy_trained' if 'policy_trained' in chunk_pca_2d else list(chunk_pca_2d.keys())[0]

    df_chunks_sampled['pca_x'] = chunk_pca_2d[primary_model]['coords'][:, 0]
    df_chunks_sampled['pca_y'] = chunk_pca_2d[primary_model]['coords'][:, 1]
    df_chunks_sampled['pca_z'] = chunk_pca_3d[primary_model]['coords'][:, 2]

    print(f"   ✓ Added PCA coordinates from {primary_model}")

    print(f"\n{'='*70}")
    print("PCA PREPARATION COMPLETE")
    print(f"{'='*70}")
    print(f"\nReady for chunk visualizations!")

else:
    print("\n⊘ Skipping PCA preparation - no chunk embeddings available")


    


CELL 9.14: PCA Preparation for Chunk Analysis

1️⃣  Preparing chunk metadata...
   ✓ Chunks: 999
   ✓ Primary topic distribution:
     Contemporary_Manifestations                  :  335 ( 33.5%)
     Historical_Slavery_Colonialism               :  485 ( 48.5%)
     Structural_Continuity_Neocolonial            :  179 ( 17.9%)

2️⃣  Performing PCA (2D and 3D)...

pretrained_bertje:
  2D PCA: 23.6% variance explained
  3D PCA: 28.2% variance explained

slavery_trained:
  2D PCA: 24.6% variance explained
  3D PCA: 31.3% variance explained

   ✓ PCA complete for 2 models

3️⃣  Adding coordinates to dataframe...
   ✓ Added PCA coordinates from pretrained_bertje

PCA PREPARATION COMPLETE

Ready for chunk visualizations!


In [34]:
# ============================================================
# CELL 9.15: CHUNK CLUSTERING 2D - Multi-Model Comparison
# ============================================================

if VIZ_AVAILABLE and len(chunk_pca_2d) > 1:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.15: Chunk Clustering 2D - Multi-Model")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Create Subplots
    # --------------------------------------------------------

    print(f"\n1️⃣  Creating multi-model comparison...")

    models_to_plot = [m for m in ['pretrained_bertje', 'slavery_trained', 'policy_trained']
                      if m in chunk_pca_2d]

    n_models = len(models_to_plot)

    fig = make_subplots(
        rows=1, cols=n_models,
        subplot_titles=[m.replace('_', ' ').title() for m in models_to_plot],
        horizontal_spacing=0.08
    )

    # Color palette
    colors = px.colors.qualitative.Set2
    if len(topics) > len(colors):
        colors = px.colors.qualitative.Alphabet
    topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

    # --------------------------------------------------------
    # 2. Plot Each Model
    # --------------------------------------------------------

    for col_idx, model_name in enumerate(models_to_plot, start=1):
        coords = chunk_pca_2d[model_name]['coords']
        var_explained = chunk_pca_2d[model_name]['var_explained']

        print(f"   Plotting {model_name}...")

        # Plot by topic
        for topic in topics:
            topic_mask = df_chunks_sampled['primary_topic'] == topic
            topic_coords = coords[topic_mask]

            # Only show legend for first subplot
            show_legend = (col_idx == 1)

            if len(topic_coords) > 0:
                fig.add_trace(
                    go.Scattergl(  # Use Scattergl for better performance with many points
                        x=topic_coords[:, 0],
                        y=topic_coords[:, 1],
                        mode='markers',
                        name=topic,
                        marker=dict(
                            size=4,
                            color=topic_colors[topic],
                            opacity=0.6,
                            line=dict(width=0)
                        ),
                        hovertemplate='<b>%{text}</b><br>Topic: ' + topic + '<extra></extra>',
                        text=[f"Chunk {i}" for i in range(len(topic_coords))],
                        showlegend=show_legend,
                        legendgroup=topic
                    ),
                    row=1, col=col_idx
                )

        # Update axes
        fig.update_xaxes(
            title_text=f"PC1 ({var_explained[0]:.1%})",
            row=1, col=col_idx
        )

        if col_idx == 1:
            fig.update_yaxes(
                title_text=f"PC2 ({var_explained[1]:.1%})",
                row=1, col=col_idx
            )

    # --------------------------------------------------------
    # 3. Update Layout
    # --------------------------------------------------------

    fig.update_layout(
        title={
            'text': "Chunk Clustering: Model Comparison (2D PCA)<br><sub>Compare how different models cluster chunks in semantic space</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        height=600,
        hovermode='closest',
        template='plotly_white',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            title="Primary Topic"
        )
    )

    # --------------------------------------------------------
    # 4. Display & Save
    # --------------------------------------------------------

    if SHOW_IN_NOTEBOOK:
        section5_figs.append(fig)  # Store for export
        fig.show()

    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'chunk_clustering_2d_multimodel.html'
        fig.write_html(str(output_path))
        print(f"   ✓ Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'chunk_clustering_2d_multimodel.png'
        fig.write_image(str(output_path), width=1800, height=800, scale=2)
        print(f"   ✓ Saved static: {output_path.name}")

    print(f"\n{'='*70}")

elif VIZ_AVAILABLE and len(chunk_pca_2d) == 1:
    print("\n⊘ Only 1 model available - skipping multi-model comparison")
else:
    print("\n⊘ Skipping chunk clustering 2D - no embeddings available")


VISUALIZATION 9.15: Chunk Clustering 2D - Multi-Model

1️⃣  Creating multi-model comparison...
   Plotting pretrained_bertje...
   Plotting slavery_trained...


   ✓ Saved interactive: chunk_clustering_2d_multimodel.html



In [35]:
# ============================================================
# CELL 9.16: CHUNK CLUSTERING 3D - Interactive Exploration
# ============================================================

if VIZ_AVAILABLE and len(chunk_pca_3d) > 0:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.16: Chunk Clustering 3D - Exploration")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Choose Model for 3D Visualization
    # --------------------------------------------------------

    model_3d = 'policy_trained' if 'policy_trained' in chunk_pca_3d else list(chunk_pca_3d.keys())[0]

    print(f"\n1️⃣  Using model: {model_3d}")

    coords = chunk_pca_3d[model_3d]['coords']
    var_explained = chunk_pca_3d[model_3d]['var_explained']

    print(f"   Variance explained: {var_explained.sum():.1%}")

    # --------------------------------------------------------
    # 2. Create 3D Scatter
    # --------------------------------------------------------

    print(f"\n2️⃣  Creating 3D visualization...")

    # Color palette
    colors = px.colors.qualitative.Set2
    if len(topics) > len(colors):
        colors = px.colors.qualitative.Alphabet
    topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

    fig = go.Figure()

    # Plot by topic
    for topic in topics:
        topic_mask = df_chunks_sampled['primary_topic'] == topic
        topic_data = df_chunks_sampled[topic_mask]
        topic_coords = coords[topic_mask]

        if len(topic_coords) > 0:
            # Prepare hover text
            hover_texts = []
            for idx, row in topic_data.iterrows():
                text_parts = [f"<b>Primary: {topic}</b>"]
                text_parts.append(f"Max Score: {row['max_score']:.2f}")
                text_parts.append(f"Multi-label: {row['n_topics_above_threshold']} topics")

                # Add top 3 scores
                chunk_scores = row[score_cols].sort_values(ascending=False).head(3)
                text_parts.append("<br>Top scores:")
                for score_col, score in chunk_scores.items():
                    topic_name = score_col.replace('score_', '')
                    text_parts.append(f"  {topic_name}: {score:.2f}")

                hover_texts.append('<br>'.join(text_parts))

            fig.add_trace(go.Scatter3d(
                x=topic_coords[:, 0],
                y=topic_coords[:, 1],
                z=topic_coords[:, 2],
                mode='markers',
                name=topic,
                marker=dict(
                    size=3,
                    color=topic_colors[topic],
                    opacity=0.7,
                    line=dict(width=0)
                ),
                hovertext=hover_texts,
                hoverinfo='text',
                showlegend=True
            ))

    # --------------------------------------------------------
    # 3. Update Layout
    # --------------------------------------------------------

    fig.update_layout(
        title={
            'text': f"Chunk Clustering - 3D Interactive Exploration<br><sub>Model: {model_3d} | Variance: {var_explained.sum():.1%}</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        scene=dict(
            xaxis_title=f"PC1 ({var_explained[0]:.1%})",
            yaxis_title=f"PC2 ({var_explained[1]:.1%})",
            zaxis_title=f"PC3 ({var_explained[2]:.1%})",
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.3)
            )
        ),
        height=800,
        template='plotly_white',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=0,
            title="Primary Topic"
        )
    )

    # --------------------------------------------------------
    # 4. Display & Save
    # --------------------------------------------------------

    if SHOW_IN_NOTEBOOK:
        section5_figs.append(fig)  # Store for export
        fig.show()

    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'chunk_clustering_3d.html'
        fig.write_html(str(output_path))
        print(f"   ✓ Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'chunk_clustering_3d.png'
        fig.write_image(str(output_path), width=1400, height=1000, scale=2)
        print(f"   ✓ Saved static: {output_path.name}")

    print(f"\n{'='*70}")

else:
    print("\n⊘ Skipping chunk clustering 3D - no embeddings available")


VISUALIZATION 9.16: Chunk Clustering 3D - Exploration

1️⃣  Using model: pretrained_bertje
   Variance explained: 28.2%

2️⃣  Creating 3D visualization...


   ✓ Saved interactive: chunk_clustering_3d.html



In [36]:
# ============================================================
# CELL 9.17: CHUNK SHIFT ANALYSIS - Multi-Model Comparison
# ============================================================

if VIZ_AVAILABLE and len(chunk_pca_2d) >= 2:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.17: Chunk Shift Analysis - Multi-Model Comparison")
    print(f"{'='*70}")

    # Get available models
    available_models = list(chunk_pca_2d.keys())
    print(f"\nAvailable models: {', '.join(available_models)}")

    # --------------------------------------------------------
    # 1. Calculate All Pairwise Shifts
    # --------------------------------------------------------

    print(f"\n1. Calculating pairwise shift vectors...")

    all_shifts = {}

    # Calculate shifts between all pairs
    for i, model1 in enumerate(available_models):
        for model2 in available_models[i+1:]:
            pair_name = f"{model1}_to_{model2}"

            coords1 = chunk_pca_2d[model1]['coords']
            coords2 = chunk_pca_2d[model2]['coords']

            # Shift vectors
            shift_vectors = coords2 - coords1

            # Shift magnitudes
            shift_magnitudes = np.linalg.norm(shift_vectors, axis=1)

            all_shifts[pair_name] = {
                'from': model1,
                'to': model2,
                'vectors': shift_vectors,
                'magnitudes': shift_magnitudes
            }

            print(f"   {model1} -> {model2}:")
            print(f"     Mean magnitude: {shift_magnitudes.mean():.3f}")
            print(f"     Median magnitude: {np.median(shift_magnitudes):.3f}")
            print(f"     Max magnitude: {shift_magnitudes.max():.3f}")

    # --------------------------------------------------------
    # 2. Analyze Shifts by Topic (for each pair)
    # --------------------------------------------------------

    print(f"\n2. Analyzing shifts by primary topic...")

    topic_shift_data = []

    for pair_name, shift_info in all_shifts.items():
        shift_magnitudes = shift_info['magnitudes']

        for topic in topics:
            topic_mask = df_chunks_sampled['primary_topic'] == topic
            topic_shifts = shift_magnitudes[topic_mask]

            if len(topic_shifts) > 0:
                topic_shift_data.append({
                    'pair': pair_name,
                    'from_model': shift_info['from'],
                    'to_model': shift_info['to'],
                    'topic': topic,
                    'n_chunks': len(topic_shifts),
                    'mean_shift': topic_shifts.mean(),
                    'median_shift': np.median(topic_shifts),
                    'max_shift': topic_shifts.max()
                })

    df_topic_shifts = pd.DataFrame(topic_shift_data)

    # Save combined stats
    stats_path = visuals_dir / 'chunk_shift_stats_combined.csv'
    df_topic_shifts.to_csv(stats_path, index=False)
    print(f"\n   Saved: {stats_path.name}")

    # --------------------------------------------------------
    # 3. Create Multi-Model Comparison Visualization
    # --------------------------------------------------------

    print(f"\n3. Creating multi-model comparison visualization...")

    # Determine layout based on number of models
    n_models = len(available_models)

    if n_models == 2:
        # Simple before/after
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(f"Model: {available_models[0]}", f"Model: {available_models[1]}"),
            horizontal_spacing=0.12
        )
        layout_config = [(1, 1), (1, 2)]
    elif n_models == 3:
        # 1x3 layout
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=tuple(f"Model: {m}" for m in available_models),
            horizontal_spacing=0.08
        )
        layout_config = [(1, 1), (1, 2), (1, 3)]
    else:
        # 2x2 or 2x3 grid for 4+ models
        n_cols = 3 if n_models > 4 else 2
        n_rows = (n_models + n_cols - 1) // n_cols
        fig = make_subplots(
            rows=n_rows, cols=n_cols,
            subplot_titles=tuple(f"Model: {m}" for m in available_models),
            horizontal_spacing=0.08,
            vertical_spacing=0.15
        )
        layout_config = [(r+1, c+1) for r in range(n_rows) for c in range(n_cols)][:n_models]

    # Color palette
    colors = px.colors.qualitative.Set2
    if len(topics) > len(colors):
        colors = px.colors.qualitative.Alphabet
    topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

    # Plot each model
    for idx, (model_name, (row, col)) in enumerate(zip(available_models, layout_config)):
        coords = chunk_pca_2d[model_name]['coords']

        # Add scatter by topic
        for topic in topics:
            topic_mask = df_chunks_sampled['primary_topic'] == topic
            topic_data = df_chunks_sampled[topic_mask]
            topic_coords = coords[topic_mask]

            if len(topic_coords) == 0:
                continue

            showlegend = (idx == 0)  # Only show legend for first subplot

            fig.add_trace(
                go.Scatter(
                    x=topic_coords[:, 0],
                    y=topic_coords[:, 1],
                    mode='markers',
                    name=topic,
                    marker=dict(
                        size=6,
                        color=topic_colors[topic],
                        opacity=0.6
                    ),
                    legendgroup=topic,
                    showlegend=showlegend,
                    hovertemplate=(
                        f'<b>{topic}</b><br>' +
                        'Chunk %{customdata}<br>' +
                        '<extra></extra>'
                    ),
                    customdata=topic_data.index
                ),
                row=row, col=col
            )

    # Update layout
    fig.update_layout(
        title={
            'text': f"Multi-Model Chunk Distribution Comparison<br><sub>2D PCA Projections | Each model's embedding space</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        height=400 if n_models <= 3 else 600,
        template='plotly_white',
        showlegend=True,
        legend=dict(
            title="Primary Topic",
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02 if n_models <= 3 else 1.01
        )
    )

    # Update axes
    for i in range(1, len(available_models) + 1):
        fig.update_xaxes(title_text="PC1", row=layout_config[i-1][0], col=layout_config[i-1][1])
        fig.update_yaxes(title_text="PC2", row=layout_config[i-1][0], col=layout_config[i-1][1])

    # Display
    if SHOW_IN_NOTEBOOK:
        section5_figs.append(fig)  # Store for export
        fig.show()

    # Save
    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'chunk_distribution_multimodel_comparison.html'
        fig.write_html(str(output_path))
        print(f"   Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'chunk_distribution_multimodel_comparison.png'
        fig.write_image(str(output_path), width=1600, height=500 if n_models <= 3 else 800, scale=2)
        print(f"   Saved static: {output_path.name}")

    # --------------------------------------------------------
    # 4. Create Shift Magnitude Visualizations (Pairwise)
    # --------------------------------------------------------

    print(f"\n4. Creating pairwise shift visualizations...")

    for pair_name, shift_info in all_shifts.items():
        print(f"\n   Creating shift visualization: {pair_name}...")

        model1 = shift_info['from']
        model2 = shift_info['to']
        shift_vectors = shift_info['vectors']
        shift_magnitudes = shift_info['magnitudes']

        # Add to dataframe
        df_chunks_sampled[f'shift_magnitude_{pair_name}'] = shift_magnitudes
        df_chunks_sampled[f'shift_x_{pair_name}'] = shift_vectors[:, 0]
        df_chunks_sampled[f'shift_y_{pair_name}'] = shift_vectors[:, 1]

        # Create before/after visualization
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(f"Before: {model1}", f"After: {model2}"),
            horizontal_spacing=0.12
        )

        coords_before = chunk_pca_2d[model1]['coords']
        coords_after = chunk_pca_2d[model2]['coords']

        # Before
        for topic in topics:
            topic_mask = df_chunks_sampled['primary_topic'] == topic
            topic_coords = coords_before[topic_mask]

            if len(topic_coords) == 0:
                continue

            fig.add_trace(
                go.Scatter(
                    x=topic_coords[:, 0],
                    y=topic_coords[:, 1],
                    mode='markers',
                    name=topic,
                    marker=dict(size=6, color=topic_colors[topic], opacity=0.6),
                    legendgroup=topic,
                    showlegend=True
                ),
                row=1, col=1
            )

        # After
        for topic in topics:
            topic_mask = df_chunks_sampled['primary_topic'] == topic
            topic_coords = coords_after[topic_mask]

            if len(topic_coords) == 0:
                continue

            fig.add_trace(
                go.Scatter(
                    x=topic_coords[:, 0],
                    y=topic_coords[:, 1],
                    mode='markers',
                    name=topic,
                    marker=dict(size=6, color=topic_colors[topic], opacity=0.6),
                    legendgroup=topic,
                    showlegend=False
                ),
                row=1, col=2
            )

        fig.update_layout(
            title={
                'text': f"Chunk Shift Analysis: {model1} → {model2}<br><sub>Mean shift: {shift_magnitudes.mean():.3f} | Median: {np.median(shift_magnitudes):.3f}</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            height=500,
            template='plotly_white',
            showlegend=True
        )

        fig.update_xaxes(title_text="PC1", row=1, col=1)
        fig.update_yaxes(title_text="PC2", row=1, col=1)
        fig.update_xaxes(title_text="PC1", row=1, col=2)
        fig.update_yaxes(title_text="PC2", row=1, col=2)

        # Display
        if SHOW_IN_NOTEBOOK:
            section5_figs.append(fig)  # Store for export
            fig.show()

        # Save
        if SAVE_INTERACTIVE:
            output_path = visuals_dir / f'chunk_shift_{pair_name}.html'
            fig.write_html(str(output_path))
            print(f"     Saved: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / f'chunk_shift_{pair_name}.png'
            fig.write_image(str(output_path), width=1400, height=600, scale=2)

    # --------------------------------------------------------
    # 5. Summary Statistics
    # --------------------------------------------------------

    print(f"\n{'='*70}")
    print("CHUNK SHIFT ANALYSIS SUMMARY")
    print(f"{'='*70}")

    for pair_name, shift_info in all_shifts.items():
        print(f"\n{shift_info['from']} -> {shift_info['to']}:")
        print(f"  Mean shift: {shift_info['magnitudes'].mean():.3f}")
        print(f"  Median shift: {np.median(shift_info['magnitudes']):.3f}")
        print(f"  Max shift: {shift_info['magnitudes'].max():.3f}")

        # Top shifting topics
        pair_topic_stats = df_topic_shifts[df_topic_shifts['pair'] == pair_name].sort_values('mean_shift', ascending=False)
        print(f"\n  Top 3 topics by shift magnitude:")
        for _, row in pair_topic_stats.head(3).iterrows():
            print(f"    {row['topic'][:40]:40s}: {row['mean_shift']:.3f}")

    print(f"\n{'='*70}")

elif VIZ_AVAILABLE:
    print("\n Skipping chunk shift analysis - need at least 2 models with chunk embeddings")
else:
    print("\n Skipping chunk shift analysis - VIZ_AVAILABLE = False")



VISUALIZATION 9.17: Chunk Shift Analysis - Multi-Model Comparison

Available models: pretrained_bertje, slavery_trained

1. Calculating pairwise shift vectors...
   pretrained_bertje -> slavery_trained:
     Mean magnitude: 6.183
     Median magnitude: 6.203
     Max magnitude: 12.375

2. Analyzing shifts by primary topic...

   Saved: chunk_shift_stats_combined.csv

3. Creating multi-model comparison visualization...


   Saved interactive: chunk_distribution_multimodel_comparison.html

4. Creating pairwise shift visualizations...

   Creating shift visualization: pretrained_bertje_to_slavery_trained...


     Saved: chunk_shift_pretrained_bertje_to_slavery_trained.html

CHUNK SHIFT ANALYSIS SUMMARY

pretrained_bertje -> slavery_trained:
  Mean shift: 6.183
  Median shift: 6.203
  Max shift: 12.375

  Top 3 topics by shift magnitude:
    Structural_Continuity_Neocolonial       : 6.966
    Historical_Slavery_Colonialism          : 6.206
    Contemporary_Manifestations             : 5.729



In [37]:
# ============================================================
# CELL 9.18: CHUNK SHIFT VECTORS - Top Shifters
# ============================================================

if VIZ_AVAILABLE and 'shift_magnitude' in df_chunks_sampled.columns:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.18: Chunk Shift Vectors - Top Shifters")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Identify Top Shifters
    # --------------------------------------------------------

    print(f"\n1️⃣  Identifying top {TOP_N_SHIFTERS} shifters...")

    df_sorted = df_chunks_sampled.sort_values('shift_magnitude', ascending=False)
    df_top_shifters = df_sorted.head(TOP_N_SHIFTERS)

    print(f"   ✓ Selected top {len(df_top_shifters)} chunks")
    print(f"   Shift magnitude range: {df_top_shifters['shift_magnitude'].min():.3f} - {df_top_shifters['shift_magnitude'].max():.3f}")

    # --------------------------------------------------------
    # 2. Create Shift Vector Visualization
    # --------------------------------------------------------

    print(f"\n2️⃣  Creating shift vector visualization...")

    coords_pre = chunk_pca_2d['pretrained_bertje']['coords']
    coords_post = chunk_pca_2d['policy_trained']['coords']

    fig = go.Figure()

    # Color palette
    colors = px.colors.qualitative.Set2
    if len(topics) > len(colors):
        colors = px.colors.qualitative.Alphabet
    topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

    # Plot all chunks (background, faded)
    for topic in topics:
        topic_mask = df_chunks_sampled['primary_topic'] == topic
        topic_coords = coords_post[topic_mask]

        if len(topic_coords) > 0:
            fig.add_trace(go.Scattergl(
                x=topic_coords[:, 0],
                y=topic_coords[:, 1],
                mode='markers',
                name=f"{topic} (background)",
                marker=dict(size=3, color=topic_colors[topic], opacity=0.2),
                showlegend=False,
                hoverinfo='skip'
            ))

    # Plot shift arrows for top shifters
    for idx, row in df_top_shifters.iterrows():
        topic = row['primary_topic']

        # Get pre/post coordinates for this chunk
        chunk_idx = df_chunks_sampled.index.get_loc(idx)
        x_pre, y_pre = coords_pre[chunk_idx]
        x_post, y_post = coords_post[chunk_idx]

        # Add arrow
        fig.add_annotation(
            x=x_post,
            y=y_post,
            ax=x_pre,
            ay=y_pre,
            xref='x',
            yref='y',
            axref='x',
            ayref='y',
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor=topic_colors[topic],
            opacity=0.7
        )

    # Add scatter for top shifters (start and end points)
    for topic in topics:
        topic_shifters = df_top_shifters[df_top_shifters['primary_topic'] == topic]

        if len(topic_shifters) > 0:
            shifter_indices = [df_chunks_sampled.index.get_loc(idx) for idx in topic_shifters.index]

            # Start points (pretrained)
            fig.add_trace(go.Scatter(
                x=coords_pre[shifter_indices, 0],
                y=coords_pre[shifter_indices, 1],
                mode='markers',
                name=f"{topic} (pre)",
                marker=dict(size=6, color=topic_colors[topic], symbol='circle-open', line=dict(width=2)),
                showlegend=True,
                legendgroup=topic,
                hovertemplate=f'<b>{topic}</b><br>Before training<extra></extra>'
            ))

            # End points (policy-trained)
            fig.add_trace(go.Scatter(
                x=coords_post[shifter_indices, 0],
                y=coords_post[shifter_indices, 1],
                mode='markers',
                name=f"{topic} (post)",
                marker=dict(size=8, color=topic_colors[topic], symbol='circle'),
                showlegend=False,
                legendgroup=topic,
                hovertemplate=f'<b>{topic}</b><br>After training<br>Shift: %{{customdata:.3f}}<extra></extra>',
                customdata=topic_shifters['shift_magnitude'].values
            ))

    # Update layout
    var_post = chunk_pca_2d['policy_trained']['var_explained']

    fig.update_layout(
        title={
            'text': f"Chunk Shift Vectors: Top {TOP_N_SHIFTERS} Largest Shifts<br><sub>Arrows show movement from pretrained (open circle) to policy-trained (filled circle)</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title=f"PC1 ({var_post[0]:.1%})",
        yaxis_title=f"PC2 ({var_post[1]:.1%})",
        height=800,
        template='plotly_white',
        hovermode='closest',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            title="Topic"
        )
    )

    # --------------------------------------------------------
    # 3. Display & Save
    # --------------------------------------------------------

    if SHOW_IN_NOTEBOOK:
        section5_figs.append(fig)  # Store for export
        fig.show()

    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'chunk_shift_vectors.html'
        fig.write_html(str(output_path))
        print(f"   ✓ Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'chunk_shift_vectors.png'
        fig.write_image(str(output_path), width=1400, height=1000, scale=2)
        print(f"   ✓ Saved static: {output_path.name}")

    print(f"\n{'='*70}")

else:
    print("\n⊘ Skipping shift vectors - shift analysis not available")


⊘ Skipping shift vectors - shift analysis not available


In [38]:
# ============================================================
# CELL 9.19: SCORE DISTRIBUTION BY TOPIC (Violin Plots)
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.19: Score Distribution by Topic")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Prepare Score Data
    # --------------------------------------------------------

    print(f"\n1️⃣  Preparing score data...")

    # Use full df_cosine (not just sampled chunks)
    print(f"   Using {len(df_cosine)} chunks (full dataset)")

    # --------------------------------------------------------
    # 2. Create Violin Plot
    # --------------------------------------------------------

    print(f"\n2️⃣  Creating violin plot...")

    fig = go.Figure()

    # Color palette
    colors = px.colors.qualitative.Set2
    if len(topics) > len(colors):
        colors = px.colors.qualitative.Alphabet
    topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

    for topic in topics:
        score_col = f'score_{topic}'

        if score_col in df_cosine.columns:
            scores = df_cosine[score_col].values

            fig.add_trace(go.Violin(
                y=scores,
                name=topic,
                box_visible=True,
                meanline_visible=True,
                fillcolor=topic_colors[topic],
                opacity=0.6,
                x0=topic[:30]  # Shortened name for x-axis
            ))

    # Update layout
    fig.update_layout(
        title={
            'text': "Score Distribution by Topic<br><sub>Violin plot showing score distribution (cosine similarity) for each topic</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        yaxis_title="Cosine Score",
        xaxis_title="Topic",
        height=700,
        template='plotly_white',
        showlegend=False
    )

    # Add threshold line
    fig.add_hline(
        y=MIN_SCORE_THRESHOLD,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Threshold ({MIN_SCORE_THRESHOLD})",
        annotation_position="right"
    )

    # --------------------------------------------------------
    # 3. Display & Save
    # --------------------------------------------------------

    if SHOW_IN_NOTEBOOK:
        section5_figs.append(fig)  # Store for export
        fig.show()

    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'score_distribution_by_topic.html'
        fig.write_html(str(output_path))
        print(f"   ✓ Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'score_distribution_by_topic.png'
        fig.write_image(str(output_path), width=1400, height=900, scale=2)
        print(f"   ✓ Saved static: {output_path.name}")

    # --------------------------------------------------------
    # 4. Statistical Summary
    # --------------------------------------------------------

    print(f"\n3️⃣  Statistical summary by topic...")

    score_stats = []

    for topic in topics:
        score_col = f'score_{topic}'

        if score_col in df_cosine.columns:
            scores = df_cosine[score_col].values

            score_stats.append({
                'topic': topic,
                'mean': scores.mean(),
                'median': np.median(scores),
                'std': scores.std(),
                'min': scores.min(),
                'max': scores.max(),
                'n_above_threshold': (scores >= MIN_SCORE_THRESHOLD).sum(),
                'pct_above_threshold': (scores >= MIN_SCORE_THRESHOLD).mean() * 100
            })

    df_score_stats = pd.DataFrame(score_stats)

    print(f"\n   Score statistics:")
    print(f"   {'Topic':<45s} {'Mean':>8s} {'Median':>8s} {'Std':>8s} {'Above Threshold':>15s}")
    print(f"   {'-'*95}")

    for _, row in df_score_stats.iterrows():
        print(f"   {row['topic'][:45]:45s} {row['mean']:8.3f} {row['median']:8.3f} {row['std']:8.3f} {row['n_above_threshold']:8d} ({row['pct_above_threshold']:5.1f}%)")

    # Save statistics
    stats_path = visuals_dir / 'score_statistics_by_topic.csv'
    df_score_stats.to_csv(stats_path, index=False)
    print(f"\n   ✓ Saved statistics: {stats_path.name}")

    print(f"\n{'='*70}")

else:
    print("\n⊘ Skipping score distribution - VIZ_AVAILABLE = False")


VISUALIZATION 9.19: Score Distribution by Topic

1️⃣  Preparing score data...
   Using 2840 chunks (full dataset)

2️⃣  Creating violin plot...


   ✓ Saved interactive: score_distribution_by_topic.html

3️⃣  Statistical summary by topic...

   Score statistics:
   Topic                                             Mean   Median      Std Above Threshold
   -----------------------------------------------------------------------------------------------
   Contemporary_Manifestations                      4.568    4.530    1.173     2840 (100.0%)
   Historical_Slavery_Colonialism                   4.825    4.775    1.221     2840 (100.0%)
   Structural_Continuity_Neocolonial                4.245    4.234    0.907     2840 (100.0%)

   ✓ Saved statistics: score_statistics_by_topic.csv



In [39]:
# ============================================================
# CELL 9.20: MULTI-LABEL DISTRIBUTION ANALYSIS
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.20: Multi-Label Distribution Analysis")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Calculate Multi-Label Statistics
    # --------------------------------------------------------

    print(f"\n1️⃣  Calculating multi-label statistics...")

    threshold = MIN_SCORE_THRESHOLD

    # Count how many topics each chunk belongs to
    df_cosine['n_topics_above_threshold'] = (df_cosine[score_cols] >= threshold).sum(axis=1)

    # Distribution
    multilabel_dist = df_cosine['n_topics_above_threshold'].value_counts().sort_index()

    print(f"\n   Multi-label distribution (threshold={threshold}):")
    for n_topics, count in multilabel_dist.items():
        pct = count / len(df_cosine) * 100
        print(f"     {n_topics} topic(s): {count:6d} chunks ({pct:5.1f}%)")

    # --------------------------------------------------------
    # 2. Create Bar Chart
    # --------------------------------------------------------

    print(f"\n2️⃣  Creating distribution chart...")

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=multilabel_dist.index,
        y=multilabel_dist.values,
        marker_color='steelblue',
        text=multilabel_dist.values,
        textposition='outside',
        hovertemplate='<b>%{x} topic(s)</b><br>Chunks: %{y}<br>Percentage: %{customdata:.1f}%<extra></extra>',
        customdata=[(v / len(df_cosine) * 100) for v in multilabel_dist.values]
    ))

    # Update layout
    fig.update_layout(
        title={
            'text': f"Multi-Label Distribution: Chunks by Number of Topics<br><sub>Threshold: {threshold} | How many chunks belong to multiple topics?</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title="Number of Topics Above Threshold",
        yaxis_title="Number of Chunks",
        height=600,
        template='plotly_white',
        showlegend=False
    )

    # --------------------------------------------------------
    # 3. Display & Save
    # --------------------------------------------------------

    if SHOW_IN_NOTEBOOK:
        section5_figs.append(fig)  # Store for export
        fig.show()

    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'multilabel_distribution.html'
        fig.write_html(str(output_path))
        print(f"   ✓ Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'multilabel_distribution.png'
        fig.write_image(str(output_path), width=1200, height=800, scale=2)
        print(f"   ✓ Saved static: {output_path.name}")

    # --------------------------------------------------------
    # 4. Multi-Label Summary
    # --------------------------------------------------------

    print(f"\n3️⃣  Multi-label summary...")

    n_single_label = (df_cosine['n_topics_above_threshold'] == 1).sum()
    n_multi_label = (df_cosine['n_topics_above_threshold'] > 1).sum()
    n_no_label = (df_cosine['n_topics_above_threshold'] == 0).sum()

    pct_single = n_single_label / len(df_cosine) * 100
    pct_multi = n_multi_label / len(df_cosine) * 100
    pct_no = n_no_label / len(df_cosine) * 100

    print(f"\n   Single-label chunks:  {n_single_label:6d} ({pct_single:5.1f}%)")
    print(f"   Multi-label chunks:   {n_multi_label:6d} ({pct_multi:5.1f}%)")
    print(f"   No-label chunks:      {n_no_label:6d} ({pct_no:5.1f}%)")

    print(f"\n{'='*70}")
    print("CHUNK SCORING ANALYSIS COMPLETE")
    print(f"{'='*70}")
    print(f"\n✅ Section 5 Complete - Chunk Analysis Done!")

else:
    print("\n⊘ Skipping multi-label analysis - VIZ_AVAILABLE = False")


VISUALIZATION 9.20: Multi-Label Distribution Analysis

1️⃣  Calculating multi-label statistics...

   Multi-label distribution (threshold=0.3):
     3 topic(s):   2840 chunks (100.0%)

2️⃣  Creating distribution chart...


   ✓ Saved interactive: multilabel_distribution.html

3️⃣  Multi-label summary...

   Single-label chunks:       0 (  0.0%)
   Multi-label chunks:     2840 (100.0%)
   No-label chunks:           0 (  0.0%)

CHUNK SCORING ANALYSIS COMPLETE

✅ Section 5 Complete - Chunk Analysis Done!


In [45]:
# ============================================================
# SECTION 5: EXPORT VISUALIZATIONS
# ============================================================
from pathlib import Path

# Create exports directory if it doesn't exist
export_dir = Path(SOURCE_WORKFLOW) / 'Visualizations'
export_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "="*70)
print(f"EXPORTING SECTION 5 VISUALIZATIONS")
print("="*70)

# Section 5 visualizations:
# 1. Multi-Topic Discovery heatmaps (score intensity by topic)
# 2. 3D Multi-Topic Clustering
# 3. Multi-Model Chunk Distribution Comparison (2D PCA projections)
# 4. Chunk Shift Vectors (pre-trained vs policy-trained)
# 5. Score Distribution by Topic (violin plots)
# 6. Multi-Label Distribution (histogram)

try:
    # Check if figures were stored in section5_figs list
    if 'section5_figs' in globals() and len(section5_figs) > 0:
        # Export each figure
        fig_names = [
        'section5_multitopic_discovery_heatmaps',
        'section5_3d_multitopic_clustering',
        'section5_multimodel_chunk_comparison',
        'section5_chunk_shift_vectors',
        'section5_score_distribution_violins',
        'section5_multilabel_distribution',
        ]

        for i, (fig, name) in enumerate(zip(section5_figs, fig_names)):
            # Save as HTML (interactive)
            html_path = export_dir / f'{name}.html'
            fig.write_html(str(html_path))
            print(f"✓ Saved: {html_path}")

            # Save as PNG (static image) - requires kaleido package
            try:
                png_path = export_dir / f'{name}.png'
                # Adjust size based on visualization type
                if '3d' in name or 'matrix' in name:
                    fig.write_image(str(png_path), width=1000, height=1000, scale=2)
                elif 'comparison' in name or 'heatmaps' in name:
                    fig.write_image(str(png_path), width=1400, height=800, scale=2)
                else:
                    fig.write_image(str(png_path), width=1200, height=800, scale=2)
                print(f"✓ Saved: {png_path}")
            except Exception as e:
                print(f"⚠️  PNG export failed: {e}")
                print(f"   (Install kaleido: pip install -U kaleido)")

        print(f"\n✓ Exported {len(section5_figs)} visualizations from Section 5")
    else:
        print(f"\nℹ️  No figures found in 'section5_figs' variable.")
        print("    To enable export, figures need to be stored in the list.")

except Exception as e:
    print(f"⚠️  Export error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)
print(f"Section 5 export complete.")
print("="*70)



EXPORTING SECTION 5 VISUALIZATIONS

ℹ️  No figures found in 'section5_figs' variable.
    To enable export, figures need to be stored in the list.

Section 5 export complete.


---
## SECTION 6: Score Distribution Analysis
---

**Goal**: Compare cosine-based vs. BERTJE-based scoring patterns.

**Key Questions**:
1. How do base cosine scores compare to BERTJE predictions?
2. Are there systematic differences between scoring methods?
3. Which chunks show largest disagreements?

**Visualizations**:
- **9.21**: Cosine vs. BERTJE Score Comparison (Scatter Matrix)
- **9.22**: Score Agreement Analysis (Correlation Heatmap)

**Note**: This section requires BERTJE predictions. If not available, it will be skipped.

In [ ]:
# ============================================================
# Initialize figure storage for Section 6
# ============================================================
section6_figs = []
print(f'Initialized section6_figs list for storing visualizations')


In [82]:
# ============================================================
# CELL 9.21: Cosine vs. BERTJE Score Comparison
# ============================================================

if VIZ_AVAILABLE:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.21: Cosine vs. BERTJE Score Comparison")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Check if BERTJE Data Already Loaded
    # --------------------------------------------------------

    print(f"\n1. Checking for BERTJE data in df_cosine...")

    # Check if BERTJE columns exist in df_cosine (from 3-way merge in Cell 9.3)
    bertje_columns = [c for c in df_cosine.columns if '_bertje' in c or 'predicted_label' in c or 'bertje_' in c.lower()]

    bertje_available = len(bertje_columns) > 0

    if bertje_available:
        print(f"   Found {len(bertje_columns)} BERTJE-related columns in df_cosine")
        print(f"   Columns: {', '.join(bertje_columns[:5])}{'...' if len(bertje_columns) > 5 else ''}")
        print(f"   Data already merged in Cell 9.3 (3-way merge)")
    else:
        print(f"   No BERTJE data found in df_cosine")
        print(f"   This is expected if bertje_labeled_corpus.csv was not available")

    if not bertje_available:
        print(f"\n   Skipping BERTJE comparison - predictions not available")
        print(f"   To enable: Ensure Bertje_labeling/bertje_labeled_corpus.csv exists in SOURCE_WORKFLOW")
    else:
        # --------------------------------------------------------
        # 2. Identify Score Column Pairs
        # --------------------------------------------------------

        print(f"\n2. Identifying cosine vs BERTJE score pairs...")

        # Find pairs of score columns for each topic
        topic_pairs = []

        for topic in topics:
            # Cosine score column (from scores_all_labeled.csv)
            cosine_col = f'score_{topic}'

            # Look for matching BERTJE score column
            # BERTJE uses shortened topic names (e.g., "Educational Disadvantage" instead of "Educational Disadvantage & Brain Drain")
            bertje_col = None

            # Extract first part of topic name (before '&' if present)
            topic_first_part = topic.split('&')[0].strip() if '&' in topic else topic

            # Try different naming patterns
            candidates = [
                f'bertje_score_{topic}',                    # Full topic name
                f'bertje_score_{topic_first_part}',         # Shortened topic name
                f'score_{topic}_bertje',                     # With _bertje suffix
                f'{topic}_bertje',
                f'bertje_{topic}'
            ]

            for candidate in candidates:
                if candidate in df_cosine.columns:
                    bertje_col = candidate
                    break

            # Fuzzy matching: find columns that contain key words from the topic
            if bertje_col is None:
                # Get significant words from topic (longer than 4 chars, not common words)
                topic_words = [w for w in topic_first_part.lower().split() if len(w) > 4]

                for col in bertje_columns:
                    col_lower = col.lower()
                    # Check if column contains all significant topic words
                    if all(word in col_lower for word in topic_words):
                        bertje_col = col
                        break

            if cosine_col in df_cosine.columns and bertje_col is not None:
                topic_pairs.append({
                    'topic': topic,
                    'cosine_col': cosine_col,
                    'bertje_col': bertje_col
                })
                print(f"   {topic}:")
                print(f"     Cosine: {cosine_col}")
                print(f"     BERTJE: {bertje_col}")
            elif cosine_col not in df_cosine.columns:
                print(f"   {topic}: No cosine column found")
            else:
                print(f"   {topic}: No matching BERTJE column found")
                print(f"     Tried: {candidates[:3]}")
                print(f"     Available BERTJE cols: {bertje_columns[:5]}")

        if len(topic_pairs) == 0:
            print(f"\n   ERROR: No valid cosine-BERTJE pairs found")
            bertje_available = False
        else:
            print(f"\n   Found {len(topic_pairs)} topic pairs for comparison")

    if bertje_available and len(topic_pairs) > 0:
        # --------------------------------------------------------
        # 3. Calculate Correlation Statistics
        # --------------------------------------------------------

        print(f"\n3. Calculating correlation statistics...")

        correlation_stats = []

        for pair in topic_pairs:
            topic = pair['topic']
            cosine_col = pair['cosine_col']
            bertje_col = pair['bertje_col']

            # Get data (drop NaN)
            data = df_cosine[[cosine_col, bertje_col]].dropna()

            if len(data) < 10:
                print(f"   {topic}: Insufficient data ({len(data)} rows)")
                continue

            # Pearson correlation
            pearson_corr = data[cosine_col].corr(data[bertje_col], method='pearson')

            # Spearman correlation
            spearman_corr = data[cosine_col].corr(data[bertje_col], method='spearman')

            # Agreement metrics
            # High score agreement: both > threshold
            threshold = MIN_SCORE_THRESHOLD
            high_cosine = data[cosine_col] > threshold
            high_bertje = data[bertje_col] > threshold
            agreement_high = (high_cosine & high_bertje).sum()
            agreement_low = (~high_cosine & ~high_bertje).sum()
            disagreement = (high_cosine != high_bertje).sum()
            agreement_pct = (agreement_high + agreement_low) / len(data) * 100

            correlation_stats.append({
                'topic': topic,
                'n_chunks': len(data),
                'pearson_r': pearson_corr,
                'spearman_r': spearman_corr,
                'agreement_pct': agreement_pct,
                'agree_both_high': agreement_high,
                'agree_both_low': agreement_low,
                'disagree': disagreement
            })

            print(f"   {topic:40s}: r={pearson_corr:.3f}, agreement={agreement_pct:.1f}%")

        df_correlations = pd.DataFrame(correlation_stats)

        # Save correlation stats
        corr_path = visuals_dir / 'cosine_bertje_correlations.csv'
        df_correlations.to_csv(corr_path, index=False)
        print(f"\n   Saved: {corr_path.name}")

        # --------------------------------------------------------
        # 4. Create Scatter Matrix Visualization
        # --------------------------------------------------------

        print(f"\n4. Creating scatter matrix visualization...")

        # Limit to top N topics for readability
        max_topics_in_viz = min(len(topic_pairs), 6)
        pairs_to_plot = topic_pairs[:max_topics_in_viz]

        n_plots = len(pairs_to_plot)
        n_cols = min(3, n_plots)
        n_rows = (n_plots + n_cols - 1) // n_cols

        fig = make_subplots(
            rows=n_rows,
            cols=n_cols,
            subplot_titles=[pair['topic'] for pair in pairs_to_plot],
            vertical_spacing=0.12,
            horizontal_spacing=0.10
        )

        for idx, pair in enumerate(pairs_to_plot):
            row = idx // n_cols + 1
            col = idx % n_cols + 1

            topic = pair['topic']
            cosine_col = pair['cosine_col']
            bertje_col = pair['bertje_col']

            # Get data
            data = df_cosine[[cosine_col, bertje_col]].dropna()

            # Sample if too many points
            if len(data) > 2000:
                data = data.sample(n=2000, random_state=42)

            fig.add_trace(
                go.Scatter(
                    x=data[cosine_col],
                    y=data[bertje_col],
                    mode='markers',
                    marker=dict(size=4, opacity=0.4, color='blue'),
                    showlegend=False,
                    hovertemplate=(
                        f'<b>{topic}</b><br>' +
                        'Cosine: %{x:.3f}<br>' +
                        'BERTJE: %{y:.3f}<br>' +
                        '<extra></extra>'
                    )
                ),
                row=row, col=col
            )

            # Add diagonal reference line
            max_val = max(data[cosine_col].max(), data[bertje_col].max())
            fig.add_trace(
                go.Scatter(
                    x=[0, max_val],
                    y=[0, max_val],
                    mode='lines',
                    line=dict(color='red', dash='dash', width=1),
                    showlegend=False,
                    hoverinfo='skip'
                ),
                row=row, col=col
            )

            # Update axes
            fig.update_xaxes(title_text="Cosine Score", row=row, col=col, range=[0, 1])
            fig.update_yaxes(title_text="BERTJE Score", row=row, col=col, range=[0, 1])

        fig.update_layout(
            title={
                'text': f"Cosine vs. BERTJE Score Comparison<br><sub>Scatter plots by topic | Red line = perfect agreement</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            height=300 * n_rows,
            template='plotly_white'
        )

        # Display
        if SHOW_IN_NOTEBOOK:
            section6_figs.append(fig)  # Store for export
            fig.show()

        # Save
        if SAVE_INTERACTIVE:
            output_path = visuals_dir / 'cosine_vs_bertje_comparison.html'
            fig.write_html(str(output_path))
            print(f"   Saved interactive: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / 'cosine_vs_bertje_comparison.png'
            fig.write_image(str(output_path), width=1200, height=300*n_rows, scale=2)
            print(f"   Saved static: {output_path.name}")

        # --------------------------------------------------------
        # 5. Summary Statistics
        # --------------------------------------------------------

        print(f"\n{'='*70}")
        print("COSINE VS BERTJE COMPARISON SUMMARY")
        print(f"{'='*70}")

        print(f"\nCorrelation Statistics:")
        print(f"{'Topic':<45} {'Pearson r':>10} {'Spearman r':>12} {'Agreement %':>13}")
        print(f"{'-'*80}")
        for _, row in df_correlations.iterrows():
            print(f"{row['topic']:<45} {row['pearson_r']:>10.3f} {row['spearman_r']:>12.3f} {row['agreement_pct']:>12.1f}%")

        # Overall statistics (only if we have data)
        if len(df_correlations) > 0:
            print(f"\nOverall:")
            print(f"  Mean Pearson r:  {df_correlations['pearson_r'].mean():.3f}")
            print(f"  Mean Spearman r: {df_correlations['spearman_r'].mean():.3f}")
            print(f"  Mean agreement:  {df_correlations['agreement_pct'].mean():.1f}%")
        else:
            print(f"\nOverall:")
            print(f"  No correlation data available")

        print(f"\n{'='*70}")

else:
    print("\n Skipping cosine vs BERTJE comparison - VIZ_AVAILABLE = False")



VISUALIZATION 9.21: Cosine vs. BERTJE Score Comparison

1. Checking for BERTJE data in df_cosine...
   Found 23 BERTJE-related columns in df_cosine
   Columns: file_path_bertje, chunk_uid_bertje, raw_text_bertje, sentence_count_bertje, token_count_bertje...
   Data already merged in Cell 9.3 (3-way merge)

2. Identifying cosine vs BERTJE score pairs...
   Arbeid_Afhankelijkheid:
     Cosine: score_Arbeid_Afhankelijkheid
     BERTJE: bertje_score_Arbeid_Afhankelijkheid
   Doorwerking_Continuiteit:
     Cosine: score_Doorwerking_Continuiteit
     BERTJE: bertje_score_Doorwerking_Continuiteit
   Erkenning_Verantwoordelijkheid:
     Cosine: score_Erkenning_Verantwoordelijkheid
     BERTJE: bertje_score_Erkenning_Verantwoordelijkheid
   Kennis_Herinnering:
     Cosine: score_Kennis_Herinnering
     BERTJE: bertje_score_Kennis_Herinnering
   Koninkrijks_Macht:
     Cosine: score_Koninkrijks_Macht
     BERTJE: bertje_score_Koninkrijks_Macht
   Raciale_Hierarchie:
     Cosine: score_Raciale_H

   Saved interactive: cosine_vs_bertje_comparison.html

COSINE VS BERTJE COMPARISON SUMMARY

Correlation Statistics:
Topic                                          Pearson r   Spearman r   Agreement %
--------------------------------------------------------------------------------
Arbeid_Afhankelijkheid                             0.810        0.811        100.0%
Doorwerking_Continuiteit                           0.797        0.807        100.0%
Erkenning_Verantwoordelijkheid                     0.824        0.812        100.0%
Kennis_Herinnering                                 0.869        0.865        100.0%
Koninkrijks_Macht                                  0.797        0.796        100.0%
Raciale_Hierarchie                                 0.855        0.847        100.0%
Slavernij_Historisch                               0.879        0.888        100.0%

Overall:
  Mean Pearson r:  0.833
  Mean Spearman r: 0.832
  Mean agreement:  100.0%



In [ ]:
# ============================================================
# CELL 9.22: Score Agreement Analysis (Correlation Heatmap)
# ============================================================

if VIZ_AVAILABLE and 'bertje_available' in dir() and bertje_available:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.22: Score Agreement Analysis")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Calculate Agreement Metrics
    # --------------------------------------------------------

    print(f"\n1️⃣  Calculating agreement metrics...")

    # For each chunk, determine primary topic by cosine vs. BERTJE
    cosine_primary = df_comparison[score_cols].idxmax(axis=1).str.replace('score_', '')

    # Get BERTJE primary topic
    bertje_topic_cols = [col for _, _, col in topic_pairs]
    df_bertje_scores = df_comparison[bertje_topic_cols].copy()
    df_bertje_scores.columns = [topic for topic, _, _ in topic_pairs]
    bertje_primary = df_bertje_scores.idxmax(axis=1)

    # Calculate agreement
    agreement = (cosine_primary == bertje_primary).mean()

    print(f"\n   Primary topic agreement: {agreement:.1%}")
    print(f"   ({(cosine_primary == bertje_primary).sum()} / {len(cosine_primary)} chunks)")

    # --------------------------------------------------------
    # 2. Confusion Matrix: Cosine Primary vs. BERTJE Primary
    # --------------------------------------------------------

    print(f"\n2️⃣  Creating confusion matrix...")

    # Build confusion matrix
    from sklearn.metrics import confusion_matrix

    # Filter to valid topic predictions
    valid_mask = cosine_primary.notna() & bertje_primary.notna()
    cosine_valid = cosine_primary[valid_mask]
    bertje_valid = bertje_primary[valid_mask]

    # Use topics list for labels
    conf_matrix = confusion_matrix(
        cosine_valid,
        bertje_valid,
        labels=topics
    )

    # Normalize by row (cosine predictions)
    conf_matrix_normalized = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis]

    # --------------------------------------------------------
    # 3. Create Heatmap
    # --------------------------------------------------------

    print(f"\n3️⃣  Creating agreement heatmap...")

    # Shorten labels for display
    topic_labels_short = [t[:25] + '...' if len(t) > 25 else t for t in topics]

    fig = go.Figure(data=go.Heatmap(
        z=conf_matrix_normalized,
        x=topic_labels_short,
        y=topic_labels_short,
        colorscale='Blues',
        zmin=0,
        zmax=1,
        text=np.round(conf_matrix_normalized, 2),
        texttemplate='%{text}',
        textfont={"size": 10},
        colorbar=dict(
            title="Agreement",
            len=0.9
        ),
        hovertemplate='Cosine: %{y}<br>BERTJE: %{x}<br>Agreement: %{z:.2f}<extra></extra>'
    ))

    # Update layout
    fig.update_layout(
        title={
            'text': f"Score Agreement Matrix: Cosine vs. BERTJE Primary Topic<br><sub>Rows=Cosine predictions, Cols=BERTJE predictions | Overall agreement: {agreement:.1%}</sub>",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title="BERTJE Primary Topic",
        yaxis_title="Cosine Primary Topic",
        height=700,
        template='plotly_white'
    )

    # Update axes
    fig.update_xaxes(tickangle=45)

    # --------------------------------------------------------
    # 4. Display & Save
    # --------------------------------------------------------

    if SHOW_IN_NOTEBOOK:
        section6_figs.append(fig)  # Store for export
        fig.show()

    if SAVE_INTERACTIVE:
        output_path = visuals_dir / 'score_agreement_heatmap.html'
        fig.write_html(str(output_path))
        print(f"   ✓ Saved interactive: {output_path.name}")

    if SAVE_STATIC:
        output_path = visuals_dir / 'score_agreement_heatmap.png'
        fig.write_image(str(output_path), width=1200, height=900, scale=2)
        print(f"   ✓ Saved static: {output_path.name}")

    # --------------------------------------------------------
    # 5. Disagreement Analysis
    # --------------------------------------------------------

    print(f"\n4️⃣  Analyzing disagreements...")

    # Find chunks where cosine and BERTJE disagree
    disagreements = df_comparison[cosine_primary != bertje_primary].copy()
    disagreements['cosine_primary'] = cosine_primary[cosine_primary != bertje_primary]
    disagreements['bertje_primary'] = bertje_primary[cosine_primary != bertje_primary]

    print(f"\n   Disagreements: {len(disagreements)} chunks ({len(disagreements)/len(df_comparison)*100:.1f}%)")

    # Most common disagreement pairs
    disagreement_pairs = disagreements.groupby(['cosine_primary', 'bertje_primary']).size().sort_values(ascending=False)

    print(f"\n   Top 5 disagreement patterns:")
    for (cosine_topic, bertje_topic), count in disagreement_pairs.head(5).items():
        pct = count / len(disagreements) * 100
        print(f"     Cosine: {cosine_topic[:30]:30s} → BERTJE: {bertje_topic[:30]:30s} | {count:4d} ({pct:5.1f}%)")

    # Save disagreements
    disagreements_path = visuals_dir / 'cosine_bertje_disagreements.csv'
    disagreements.to_csv(disagreements_path, index=False)
    print(f"\n   ✓ Saved disagreements: {disagreements_path.name}")

    print(f"\n{'='*70}")
    print("SCORE DISTRIBUTION ANALYSIS COMPLETE")
    print(f"{'='*70}")
    print(f"\n✅ Section 6 Complete - Score Comparison Done!")

elif VIZ_AVAILABLE:
    print("\n⊘ Skipping score agreement analysis - BERTJE predictions not available")
else:
    print("\n⊘ Skipping score agreement analysis - VIZ_AVAILABLE = False")

In [ ]:
# ============================================================
# SECTION 6: EXPORT VISUALIZATIONS
# ============================================================
from pathlib import Path

# Create exports directory if it doesn't exist
export_dir = Path(SOURCE_WORKFLOW) / 'Visualizations'
export_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "="*70)
print(f"EXPORTING SECTION 6 VISUALIZATIONS")
print("="*70)

# Section 6 visualizations:
# 1. Cosine vs BERTJE Score Comparison (scatter plots by topic)
# 2. Score Agreement Matrix (confusion matrix style heatmap)

try:
    # Check if figures were stored in section6_figs list
    if 'section6_figs' in globals() and len(section6_figs) > 0:
        # Export each figure
        fig_names = [
        'section6_cosine_vs_bertje_comparison',
        'section6_score_agreement_matrix',
        ]

        for i, (fig, name) in enumerate(zip(section6_figs, fig_names)):
            # Save as HTML (interactive)
            html_path = export_dir / f'{name}.html'
            fig.write_html(str(html_path))
            print(f"✓ Saved: {html_path}")

            # Save as PNG (static image) - requires kaleido package
            try:
                png_path = export_dir / f'{name}.png'
                # Adjust size based on visualization type
                if '3d' in name or 'matrix' in name:
                    fig.write_image(str(png_path), width=1000, height=1000, scale=2)
                elif 'comparison' in name or 'heatmaps' in name:
                    fig.write_image(str(png_path), width=1400, height=800, scale=2)
                else:
                    fig.write_image(str(png_path), width=1200, height=800, scale=2)
                print(f"✓ Saved: {png_path}")
            except Exception as e:
                print(f"⚠️  PNG export failed: {e}")
                print(f"   (Install kaleido: pip install -U kaleido)")

        print(f"\n✓ Exported {len(section6_figs)} visualizations from Section 6")
    else:
        print(f"\nℹ️  No figures found in 'section6_figs' variable.")
        print("    To enable export, figures need to be stored in the list.")

except Exception as e:
    print(f"⚠️  Export error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)
print(f"Section 6 export complete.")
print("="*70)


---
## SECTION 7: Thesis-Specific Visualizations (Optional)
---

**Goal**: Generate visualizations tailored to thesis research questions.

**Key Questions**:
1. How does slavery legacy discourse vary over time (2015-2024)?
2. Which document types emphasize which topics?
3. Are there temporal patterns in topic prominence?

**Visualizations**:
- **9.23**: Temporal Analysis - Topic Scores Over Time
- **9.24**: Document Type Analysis - Topic Distribution by Type

**Note**: These visualizations require metadata (year, doc_type). If filters were applied or metadata unavailable, these will be skipped.

In [ ]:
# ============================================================
# CELL 9.23: Temporal Analysis - Topic Scores Over Time
# ============================================================

if VIZ_AVAILABLE and 'year' in df_cosine.columns:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.23: Temporal Analysis - Topics Over Time")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Prepare Temporal Data
    # --------------------------------------------------------

    print(f"\n1️⃣  Preparing temporal data...")

    # Filter to valid years (drop NaN)
    df_temporal = df_cosine[df_cosine['year'].notna()].copy()

    # Convert year to int if needed
    if df_temporal['year'].dtype == 'object':
        df_temporal['year'] = pd.to_numeric(df_temporal['year'], errors='coerce')
        df_temporal = df_temporal[df_temporal['year'].notna()]

    df_temporal['year'] = df_temporal['year'].astype(int)

    print(f"   ✓ Chunks with year data: {len(df_temporal)}")
    print(f"   Year range: {df_temporal['year'].min()} - {df_temporal['year'].max()}")

    # Count chunks per year
    year_counts = df_temporal['year'].value_counts().sort_index()
    print(f"\n   Chunks per year:")
    for year, count in year_counts.items():
        print(f"     {year}: {count:4d} chunks")

    if len(df_temporal) < 10:
        print(f"\n   ⚠ WARNING: Too few chunks with year data ({len(df_temporal)})")
        print(f"   ⊘ Skipping temporal analysis")
    else:
        # --------------------------------------------------------
        # 2. Calculate Mean Score Per Year Per Topic
        # --------------------------------------------------------

        print(f"\n2️⃣  Calculating mean scores per year...")

        temporal_data = []

        for year in sorted(df_temporal['year'].unique()):
            year_data = df_temporal[df_temporal['year'] == year]

            for topic in topics:
                score_col = f'score_{topic}'

                if score_col in year_data.columns:
                    mean_score = year_data[score_col].mean()
                    median_score = year_data[score_col].median()
                    n_chunks = len(year_data)

                    temporal_data.append({
                        'year': year,
                        'topic': topic,
                        'mean_score': mean_score,
                        'median_score': median_score,
                        'n_chunks': n_chunks
                    })

        df_temporal_agg = pd.DataFrame(temporal_data)

        # --------------------------------------------------------
        # 3. Create Line Chart
        # --------------------------------------------------------

        print(f"\n3️⃣  Creating temporal visualization...")

        fig = go.Figure()

        # Color palette
        colors = px.colors.qualitative.Set2
        if len(topics) > len(colors):
            colors = px.colors.qualitative.Alphabet
        topic_colors = {topic: colors[i % len(colors)] for i, topic in enumerate(topics)}

        # Plot each topic
        for topic in topics:
            topic_data = df_temporal_agg[df_temporal_agg['topic'] == topic].sort_values('year')

            fig.add_trace(go.Scatter(
                x=topic_data['year'],
                y=topic_data['mean_score'],
                mode='lines+markers',
                name=topic,
                line=dict(color=topic_colors[topic], width=2),
                marker=dict(size=8),
                hovertemplate=(
                    '<b>%{fullData.name}</b><br>' +
                    'Year: %{x}<br>' +
                    'Mean Score: %{y:.3f}<br>' +
                    'N chunks: %{customdata}<br>' +
                    '<extra></extra>'
                ),
                customdata=topic_data['n_chunks']
            ))

        # Update layout
        fig.update_layout(
            title={
                'text': "Temporal Analysis: Topic Prominence Over Time (2015-2024)<br><sub>Mean cosine scores per year | IDPAD decade</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            xaxis_title="Year",
            yaxis_title="Mean Cosine Score",
            height=600,
            template='plotly_white',
            hovermode='x unified',
            legend=dict(
                orientation="v",
                yanchor="top",
                y=1,
                xanchor="left",
                x=1.02,
                title="Topics"
            )
        )

        # Add IDPAD period annotation
        fig.add_vrect(
            x0=2015, x1=2024,
            fillcolor="lightblue",
            opacity=0.1,
            layer="below",
            line_width=0,
            annotation_text="IDPAD 2015-2024",
            annotation_position="top left"
        )

        # --------------------------------------------------------
        # 4. Display & Save
        # --------------------------------------------------------

        if SHOW_IN_NOTEBOOK:
            fig.show()

        if SAVE_INTERACTIVE:
            output_path = visuals_dir / 'temporal_analysis_topics_over_time.html'
            fig.write_html(str(output_path))
            print(f"   ✓ Saved interactive: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / 'temporal_analysis_topics_over_time.png'
            fig.write_image(str(output_path), width=1400, height=800, scale=2)
            print(f"   ✓ Saved static: {output_path.name}")

        # Save temporal data
        temporal_data_path = visuals_dir / 'temporal_topic_scores.csv'
        df_temporal_agg.to_csv(temporal_data_path, index=False)
        print(f"   ✓ Saved temporal data: {temporal_data_path.name}")

        print(f"\n{'='*70}")

elif VIZ_AVAILABLE and 'year' not in df_cosine.columns:
    print("\n⊘ Skipping temporal analysis - 'year' column not available")
else:
    print("\n⊘ Skipping temporal analysis - VIZ_AVAILABLE = False")

In [ ]:
# ============================================================
# CELL 9.24: Document Type Analysis - Topics by Document Type
# ============================================================

if VIZ_AVAILABLE and 'doc_type' in df_cosine.columns:
    print(f"\n{'='*70}")
    print("VISUALIZATION 9.24: Document Type Analysis")
    print(f"{'='*70}")

    # --------------------------------------------------------
    # 1. Prepare Document Type Data
    # --------------------------------------------------------

    print(f"\n1️⃣  Preparing document type data...")

    # Filter to valid doc_types (drop NaN)
    df_doctype = df_cosine[df_cosine['doc_type'].notna()].copy()

    print(f"   ✓ Chunks with doc_type: {len(df_doctype)}")

    # Count chunks per doc_type
    doctype_counts = df_doctype['doc_type'].value_counts()
    print(f"\n   Document types:")
    for doctype, count in doctype_counts.items():
        print(f"     {doctype:30s}: {count:4d} chunks")

    if len(df_doctype) < 10 or len(doctype_counts) < 2:
        print(f"\n   ⚠ WARNING: Insufficient doc_type diversity")
        print(f"   ⊘ Skipping document type analysis")
    else:
        # --------------------------------------------------------
        # 2. Calculate Mean Score Per Doc Type Per Topic
        # --------------------------------------------------------

        print(f"\n2️⃣  Calculating mean scores per document type...")

        doctype_data = []

        for doctype in df_doctype['doc_type'].unique():
            doctype_chunks = df_doctype[df_doctype['doc_type'] == doctype]

            for topic in topics:
                score_col = f'score_{topic}'

                if score_col in doctype_chunks.columns:
                    mean_score = doctype_chunks[score_col].mean()
                    median_score = doctype_chunks[score_col].median()
                    n_chunks = len(doctype_chunks)

                    doctype_data.append({
                        'doc_type': doctype,
                        'topic': topic,
                        'mean_score': mean_score,
                        'median_score': median_score,
                        'n_chunks': n_chunks
                    })

        df_doctype_agg = pd.DataFrame(doctype_data)

        # --------------------------------------------------------
        # 3. Create Grouped Bar Chart
        # --------------------------------------------------------

        print(f"\n3️⃣  Creating document type visualization...")

        # Pivot for easier plotting
        df_pivot = df_doctype_agg.pivot(index='topic', columns='doc_type', values='mean_score')

        fig = go.Figure()

        # Add bar for each doc_type
        for doctype in df_pivot.columns:
            fig.add_trace(go.Bar(
                name=doctype,
                x=df_pivot.index,
                y=df_pivot[doctype],
                text=np.round(df_pivot[doctype], 2),
                textposition='outside',
                textfont=dict(size=9)
            ))

        # Update layout
        fig.update_layout(
            title={
                'text': "Topic Distribution by Document Type<br><sub>Mean cosine scores | Which document types emphasize which topics?</sub>",
                'x': 0.5,
                'xanchor': 'center'
            },
            xaxis_title="Topic",
            yaxis_title="Mean Cosine Score",
            barmode='group',
            height=700,
            template='plotly_white',
            xaxis_tickangle=-45,
            legend=dict(
                orientation="v",
                yanchor="top",
                y=1,
                xanchor="left",
                x=1.02,
                title="Document Type"
            )
        )

        # --------------------------------------------------------
        # 4. Display & Save
        # --------------------------------------------------------

        if SHOW_IN_NOTEBOOK:
            fig.show()

        if SAVE_INTERACTIVE:
            output_path = visuals_dir / 'doctype_analysis_topics_by_type.html'
            fig.write_html(str(output_path))
            print(f"   ✓ Saved interactive: {output_path.name}")

        if SAVE_STATIC:
            output_path = visuals_dir / 'doctype_analysis_topics_by_type.png'
            fig.write_image(str(output_path), width=1400, height=900, scale=2)
            print(f"   ✓ Saved static: {output_path.name}")

        # Save doctype data
        doctype_data_path = visuals_dir / 'doctype_topic_scores.csv'
        df_doctype_agg.to_csv(doctype_data_path, index=False)
        print(f"   ✓ Saved doctype data: {doctype_data_path.name}")

        print(f"\n{'='*70}")
        print("THESIS-SPECIFIC VISUALIZATIONS COMPLETE")
        print(f"{'='*70}")
        print(f"\n✅ Section 7 Complete!")

elif VIZ_AVAILABLE and 'doc_type' not in df_cosine.columns:
    print("\n⊘ Skipping document type analysis - 'doc_type' column not available")
else:
    print("\n⊘ Skipping document type analysis - VIZ_AVAILABLE = False")

---
## SECTION 8: Summary & Export
---

**Goal**: Summarize all visualizations and export metadata.

**Outputs**:
- Summary report of all generated visualizations
- Visualization inventory (CSV)
- Configuration snapshot (JSON)

In [ ]:
# ============================================================
# CELL 9.25: Visualization Summary & Inventory
# ============================================================

print(f"\n{'='*70}")
print("CHECKPOINT 9: VISUALIZATION SUMMARY")
print(f"{'='*70}")

# --------------------------------------------------------
# 1. List All Generated Files
# --------------------------------------------------------

print(f"\n1️⃣  Scanning generated visualizations...")

if visuals_dir.exists():
    viz_files = list(visuals_dir.glob('*'))

    # Categorize files
    html_files = [f for f in viz_files if f.suffix == '.html']
    png_files = [f for f in viz_files if f.suffix == '.png']
    csv_files = [f for f in viz_files if f.suffix == '.csv']
    other_files = [f for f in viz_files if f.suffix not in ['.html', '.png', '.csv']]

    print(f"\n   Total files generated: {len(viz_files)}")
    print(f"   - Interactive (HTML): {len(html_files)}")
    print(f"   - Static (PNG): {len(png_files)}")
    print(f"   - Data (CSV): {len(csv_files)}")
    print(f"   - Other: {len(other_files)}")

    # --------------------------------------------------------
    # 2. Create Inventory DataFrame
    # --------------------------------------------------------

    print(f"\n2️⃣  Creating visualization inventory...")

    inventory = []

    for viz_file in viz_files:
        file_info = {
            'filename': viz_file.name,
            'type': viz_file.suffix[1:],  # Remove leading dot
            'size_kb': viz_file.stat().st_size / 1024,
            'created': pd.Timestamp.fromtimestamp(viz_file.stat().st_mtime)
        }

        # Categorize by section
        name_lower = viz_file.name.lower()
        if 'weight' in name_lower or 'expansion' in name_lower or 'dict' in name_lower and 'clustering' in name_lower:
            section = 'Section 3: Dictionary Fitness'
        elif 'cluster_quality' in name_lower or 'topic_separation' in name_lower or 'training_metrics' in name_lower:
            section = 'Section 4: Topic Coherence'
        elif 'chunk' in name_lower and ('clustering' in name_lower or 'shift' in name_lower):
            section = 'Section 5: Chunk Analysis'
        elif 'score' in name_lower and ('distribution' in name_lower or 'multilabel' in name_lower):
            section = 'Section 5: Chunk Analysis'
        elif 'cosine' in name_lower and 'bertje' in name_lower:
            section = 'Section 6: Score Distribution'
        elif 'temporal' in name_lower or 'doctype' in name_lower:
            section = 'Section 7: Thesis-Specific'
        else:
            section = 'Other'

        file_info['section'] = section
        inventory.append(file_info)

    df_inventory = pd.DataFrame(inventory)
    df_inventory = df_inventory.sort_values(['section', 'filename'])

    # --------------------------------------------------------
    # 3. Print Inventory by Section
    # --------------------------------------------------------

    print(f"\n3️⃣  Visualization inventory by section:")

    for section in df_inventory['section'].unique():
        section_files = df_inventory[df_inventory['section'] == section]

        print(f"\n   {section}:")

        for _, row in section_files.iterrows():
            print(f"     - {row['filename']:50s} ({row['type']:4s}, {row['size_kb']:7.1f} KB)")

    # --------------------------------------------------------
    # 4. Save Inventory
    # --------------------------------------------------------

    inventory_path = visuals_dir / 'visualization_inventory.csv'
    df_inventory.to_csv(inventory_path, index=False)
    print(f"\n   ✓ Saved inventory: {inventory_path.name}")

else:
    print(f"\n   ⚠ Visuals directory not found: {visuals_dir}")

# --------------------------------------------------------
# 5. Summary Statistics
# --------------------------------------------------------

print(f"\n{'='*70}")
print("EXECUTION SUMMARY")
print(f"{'='*70}")

if VIZ_AVAILABLE:
    print(f"\n✅ Visualization pipeline completed successfully!")

    # Data summary
    print(f"\nData processed:")
    print(f"  - Dictionary terms: {len(df_dict)}")
    print(f"  - Topics: {len(topics)}")
    print(f"  - Chunks (full): {len(df_cosine)}")
    if 'df_chunks_sampled' in dir() and len(df_chunks_sampled) > 0:
        print(f"  - Chunks (sampled): {len(df_chunks_sampled)}")

    # Models summary
    if 'models' in dir() and models:
        print(f"\nModels used:")
        for model_name in models.keys():
            if models[model_name] is not None or model_name == 'base_cosine':
                print(f"  ✓ {model_name}")

    # Embeddings summary
    if 'dict_embeddings' in dir() and dict_embeddings:
        print(f"\nDictionary embeddings generated:")
        for model_name, emb in dict_embeddings.items():
            print(f"  - {model_name}: {emb.shape}")

    if 'chunk_embeddings' in dir() and chunk_embeddings:
        print(f"\nChunk embeddings generated:")
        for model_name, emb in chunk_embeddings.items():
            print(f"  - {model_name}: {emb.shape}")

else:
    print(f"\n⚠ Visualization pipeline did not complete")
    print(f"   VIZ_AVAILABLE = False (check error messages above)")

print(f"\n{'='*70}")

In [ ]:
# ============================================================
# CELL 9.26: Export Configuration Snapshot
# ============================================================

print(f"\n{'='*70}")
print("CELL 9.26: Exporting Configuration Snapshot")
print(f"{'='*70}")

# --------------------------------------------------------
# 1. Collect Configuration
# --------------------------------------------------------

print(f"\n1️⃣  Collecting configuration...")

config_snapshot = {
    'checkpoint': 'CP9',
    'notebook_version': 'v26',
    'timestamp': pd.Timestamp.now().isoformat(),

    # Data source
    'data_source': str(base_dir) if 'source_fs' in dir() else 'unknown',

    # Model configuration
    'models_compared': COMPARE_MODELS if 'COMPARE_MODELS' in dir() else {},
    'model_paths': MODEL_PATHS if 'MODEL_PATHS' in dir() else {},

    # Metadata filters
    'metadata_filters': METADATA_FILTERS if 'METADATA_FILTERS' in dir() else {},

    # Visualization settings
    'visualization_settings': {
        'min_score_threshold': MIN_SCORE_THRESHOLD if 'MIN_SCORE_THRESHOLD' in dir() else None,
        'top_n_shifters': TOP_N_SHIFTERS if 'TOP_N_SHIFTERS' in dir() else None,
        'sample_size_3d': SAMPLE_SIZE_3D if 'SAMPLE_SIZE_3D' in dir() else None,
        'pca_random_state': PCA_RANDOM_STATE if 'PCA_RANDOM_STATE' in dir() else None,
        'figure_dpi': FIGURE_DPI if 'FIGURE_DPI' in dir() else None
    },

    # Output settings
    'output_settings': {
        'save_interactive': SAVE_INTERACTIVE if 'SAVE_INTERACTIVE' in dir() else None,
        'save_static': SAVE_STATIC if 'SAVE_STATIC' in dir() else None,
        'show_in_notebook': SHOW_IN_NOTEBOOK if 'SHOW_IN_NOTEBOOK' in dir() else None
    },

    # Data summary
    'data_summary': {
        'n_dictionary_terms': len(df_dict) if 'df_dict' in dir() else 0,
        'n_topics': len(topics) if 'topics' in dir() else 0,
        'n_chunks_full': len(df_cosine) if 'df_cosine' in dir() else 0,
        'n_chunks_sampled': len(df_chunks_sampled) if 'df_chunks_sampled' in dir() else 0
    },

    # Execution status
    'viz_available': VIZ_AVAILABLE if 'VIZ_AVAILABLE' in dir() else False
}

# --------------------------------------------------------
# 2. Save Configuration
# --------------------------------------------------------

print(f"\n2️⃣  Saving configuration snapshot...")

config_path = visuals_dir / 'cp9_configuration.json'

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config_snapshot, f, indent=2, ensure_ascii=False, default=str)

print(f"   ✓ Saved configuration: {config_path.name}")

# --------------------------------------------------------
# 3. Print Configuration Summary
# --------------------------------------------------------

print(f"\n3️⃣  Configuration summary:")

print(f"\n   Data source: {config_snapshot['data_source']}")

print(f"\n   Models compared:")
for model, enabled in config_snapshot['models_compared'].items():
    status = '✓' if enabled else '⊘'
    print(f"     {status} {model}")

if config_snapshot['metadata_filters']['doc_type'] or    config_snapshot['metadata_filters']['year_range'] or    config_snapshot['metadata_filters']['doc_folder']:
    print(f"\n   Metadata filters applied:")
    for filter_name, filter_value in config_snapshot['metadata_filters'].items():
        if filter_value:
            print(f"     - {filter_name}: {filter_value}")
else:
    print(f"\n   No metadata filters applied (using full dataset)")

print(f"\n   Visualization settings:")
print(f"     - Threshold: {config_snapshot['visualization_settings']['min_score_threshold']}")
print(f"     - Sample size (3D): {config_snapshot['visualization_settings']['sample_size_3d']}")
print(f"     - Random state: {config_snapshot['visualization_settings']['pca_random_state']}")

print(f"\n{'='*70}")
print("CHECKPOINT 9 COMPLETE!")
print(f"{'='*70}")

print(f"\n✅ All visualizations generated successfully!")
print(f"\n📂 Output directory: {visuals_dir}")
print(f"\n🎨 Ready for thesis integration and analysis!")
print(f"\n{'='*70}")